In [24]:
# ==========================================
# 02_multi_omics_ef_lf_cv_rebuild
# EF/LF comparison + top10 late-fusion search
# ==========================================

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import Lasso, ElasticNet, Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import GroupKFold, GridSearchCV, ParameterGrid, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression

from scipy import sparse
from joblib import Parallel, delayed
import traceback


In [25]:
# ==========================================
# Config
# ==========================================

# COMBO_MAP = {
#     "A": ["prot"],
#     "B": ["met"],
#     "C": ["rna"],
#     "AB": ["prot", "met"],
#     "AC": ["prot", "rna"],
#     "BC": ["met", "rna"],
#     "ABC": ["prot", "met", "rna"],
# }

DATA_ROOT = Path("../../DifferentCom_data_rebuild")
TP_DIR = DATA_ROOT / "tp_views"
BASE_DIR = DATA_ROOT / "base_patient_tables"

EXPERIMENT_TAG = "ef_paper_style_v13_tp_model_refined_tuning"

SAVE_DIR = DATA_ROOT / f"results_multi_omics_ef_train_only_{EXPERIMENT_TAG}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_PATH = BASE_DIR / "target_by_patient.csv"

TISSUES = ["csf", "ser"]
TIMEPOINTS = [24, 48, 72, 96, 120]
COMBOS = ["ABC"]

# v13 design:
# - Keep full paper-style tissue/timepoint coverage.
# - Keep nested CV: OUTER 5-fold for OOF evaluation + INNER 3-fold for model/grid selection.
# - Interpret and select results by tissue + TP + model, not by one global mean OOF.
# - Use v12 as the exploration pass and keep only models that were useful within each tissue/TP.
# - Strong windows keep broader model/grid choices; weak serum windows stay ridge-only.
MODEL_TYPES = [
    "ridge",
    "elasticnet",
    "pls",
    "svr_linear",
    "gbr",
]

CV_SEARCH_MODE = "nested_outer5_inner3_v13_tp_model_refined_tuning"


def get_ef_tuning_group(tissue: str, tp: int) -> str:
    """
    v13 tuning strength based on EF v12 tissue + TP + model results.

    Important: this is only a tuning-budget label. Final interpretation should be
    based on tissue + TP + model summaries and best_row_by_tissue_tp.
    """
    tissue = str(tissue).lower()
    tp = int(tp)

    if (tissue == "csf" and tp in [72, 96]) or (tissue == "ser" and tp == 96):
        return "strong"
    if (tissue == "csf" and tp in [24, 48, 120]) or (tissue == "ser" and tp == 120):
        return "middle"
    return "weak"


def get_ef_model_types_for_job(tissue: str, tp: int) -> list[str]:
    """
    v13 result-based model routing from EF v12 grouped analysis.

    The goal is not to maximize one global mean OOF. Instead, each tissue/TP keeps
    the models that were actually useful for that tissue/TP.
    """
    tissue = str(tissue).lower()
    tp = int(tp)

    if tissue == "csf":
        if tp == 24:
            candidates = ["ridge", "elasticnet"]
        elif tp == 48:
            candidates = ["ridge", "svr_linear"]
        elif tp == 72:
            candidates = ["ridge", "elasticnet", "svr_linear"]
        elif tp == 96:
            candidates = ["ridge", "elasticnet", "pls", "svr_linear", "gbr"]
        elif tp == 120:
            candidates = ["ridge"]
        else:
            candidates = ["ridge"]

    elif tissue == "ser":
        if tp in [24, 48, 72]:
            candidates = ["ridge"]
        elif tp == 96:
            candidates = ["ridge", "elasticnet", "pls", "gbr"]
        elif tp == 120:
            candidates = ["ridge", "elasticnet"]
        else:
            candidates = ["ridge"]
    else:
        candidates = ["ridge"]

    return [m for m in candidates if m in MODEL_TYPES]


def get_ef_clinical_modes_for_job(tissue: str, tp: int) -> list[str]:
    """
    v13 clinical routing.

    Keep both omics_only and light for all retained model jobs so that the final
    best row is selected within each tissue/TP, matching the paper-style comparison.
    """
    return CLINICAL_MODES_TO_RUN.copy()


# Paper-style comparison:
# - omics_only: omics features only
# - light: omics + Age/Gender/Level
CLINICAL_MODES_TO_RUN = ["omics_only", "light"]

# v13 keeps f_regression_topk as the primary fold-safe ranking method.
# v10/v12 showed hybrid_topk did not improve EF OOF R2, so model choice is routed by tissue+TP.
FEATURE_SELECTION_MODES_TO_RUN = [
    "f_regression_topk",
]

USE_BLOCKWISE_EF = True
BLOCKWISE_MIN_PER_BLOCK = 5
EF_MAX_FINAL_SELECTED = 70
EF_MAX_FINAL_SELECTED_LATE = 40

MIN_FEATURE_FREQ = 3
MIN_FEATURE_FREQ_LATE = 3
LATE_TP_START = 96

USE_DELTA_VIEW = False
LIGHT_CLIN_COLS = ["Age", "Gender", "Level"]

N_JOBS_EXPERIMENT = 8
PARALLEL_BACKEND = "loky"

N_SPLITS_OUTER = 5
N_SPLITS_INNER = 3

USE_STRATIFIED_OUTER_CV = True
USE_STRATIFIED_INNER_CV = True
N_Y_BINS_FOR_OUTER_CV = 3
OUTER_CV_RANDOM_STATE = 42

LOW_VALID_SST_WARN_THRESHOLD = 200.0

USE_FEATURE_WEIGHTING = True
FEATURE_WEIGHT_MODES = ["none", "soft"] if USE_FEATURE_WEIGHTING else ["none"]

FEATURE_WEIGHT_MIN = 1.0
FEATURE_WEIGHT_MAX = 1.20
FEATURE_WEIGHT_DEFAULT = 1.0
FEATURE_WEIGHT_TOPK_HIGH = 1.20
FEATURE_WEIGHT_TOPK_LOW = 1.0

RUN_EF = True
RUN_LF = False

TOP_N = 10
COMPUTE_TRAIN_INNER_OOF = False

MIN_OBS_FRAC = 0.40
SPARSE_FALLBACK_THRESHOLDS = [0.40, 0.35]

MIN_FEATURES_AFTER_FILTER = 15
MIN_OMICS_FEATURES_AFTER_FILTER = 60

print("SAVE_DIR:", SAVE_DIR.resolve())
print("EXPERIMENT_TAG:", EXPERIMENT_TAG)
print("CV_SEARCH_MODE:", CV_SEARCH_MODE)
print("MODEL_TYPES:", MODEL_TYPES)
print("EF_TUNING_GROUPS:", {(t,tp): get_ef_tuning_group(t,tp) for t in TISSUES for tp in TIMEPOINTS})
print("EF_MODEL_ROUTING:", {(t,tp): get_ef_model_types_for_job(t,tp) for t in TISSUES for tp in TIMEPOINTS})
print("EF_CLINICAL_ROUTING:", {(t,tp): get_ef_clinical_modes_for_job(t,tp) for t in TISSUES for tp in TIMEPOINTS})
print("CLINICAL_MODES_TO_RUN:", CLINICAL_MODES_TO_RUN)
print("FEATURE_SELECTION_MODES_TO_RUN:", FEATURE_SELECTION_MODES_TO_RUN)
print("USE_FEATURE_WEIGHTING:", USE_FEATURE_WEIGHTING)
print("FEATURE_WEIGHT_MODES:", FEATURE_WEIGHT_MODES)


SAVE_DIR: /data2/jinoh001/stable/research_SPI/DifferentCom_data_rebuild/results_multi_omics_ef_train_only_ef_paper_style_v13_tp_model_refined_tuning
EXPERIMENT_TAG: ef_paper_style_v13_tp_model_refined_tuning
CV_SEARCH_MODE: nested_outer5_inner3_v13_tp_model_refined_tuning
MODEL_TYPES: ['ridge', 'elasticnet', 'pls', 'svr_linear', 'gbr']
EF_TUNING_GROUPS: {('csf', 24): 'middle', ('csf', 48): 'middle', ('csf', 72): 'strong', ('csf', 96): 'strong', ('csf', 120): 'middle', ('ser', 24): 'weak', ('ser', 48): 'weak', ('ser', 72): 'weak', ('ser', 96): 'strong', ('ser', 120): 'middle'}
EF_MODEL_ROUTING: {('csf', 24): ['ridge', 'elasticnet'], ('csf', 48): ['ridge', 'svr_linear'], ('csf', 72): ['ridge', 'elasticnet', 'svr_linear'], ('csf', 96): ['ridge', 'elasticnet', 'pls', 'svr_linear', 'gbr'], ('csf', 120): ['ridge'], ('ser', 24): ['ridge'], ('ser', 48): ['ridge'], ('ser', 72): ['ridge'], ('ser', 96): ['ridge', 'elasticnet', 'pls', 'gbr'], ('ser', 120): ['ridge', 'elasticnet']}
EF_CLINICAL_ROUT

In [26]:
def read_id_txt(path: str) -> list[str]:
    df = pd.read_csv(path)
    return df["Patient"].astype(str).tolist()




In [27]:

TRAIN_ID_PATH = "../../data/training_id.txt"
TRAIN_IDS = read_id_txt(TRAIN_ID_PATH)


In [28]:
# ==========================================
# Loader / helper
# ==========================================
def load_feature_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        index_col=0,
        na_values=["", " ", "NA", "N/A", "nan", "NaN", ".", "-"],
        keep_default_na=True,
    )
    df.index = df.index.astype(str)

    # whitespace-only string도 NaN 처리
    df = df.replace(r"^\s*$", np.nan, regex=True)

    sentinel_values = [
        -10.210340372,
        -1.127439639,
        -1.128465252,
    ]
    df = df.replace(sentinel_values, np.nan)

    # clinical columns는 merge_with_light_clinical에서 붙으므로,
    # 여기 들어온 feature CSV는 기본적으로 omics라고 보고 numeric coercion.
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def load_target(path: Path) -> pd.Series:
    y = pd.read_csv(path, index_col=0).iloc[:, 0]
    y.index = y.index.astype(str)
    y.name = "DeltaTMS"
    return y


def build_feature_path(tissue: str, combo: str, tp: int, use_delta: bool = False) -> Path:
    if use_delta:
        return TP_DIR / f"x_omicsdelta_{tissue}_{combo}_{tp}.csv"
    return TP_DIR / f"x_omics_{tissue}_{combo}_{tp}.csv"


def build_clinical_path(tissue: str, combo: str, tp: int) -> Path:
    return TP_DIR / f"x_clinical_{tissue}_{combo}_{tp}.csv"


def load_light_clinical_csv(path: Path) -> pd.DataFrame:
    clin = pd.read_csv(path, index_col=0)
    clin.index = clin.index.astype(str)

    missing_cols = [c for c in LIGHT_CLIN_COLS if c not in clin.columns]
    if len(missing_cols) > 0:
        raise ValueError(f"clinical file missing required cols: {missing_cols}")

    return clin[LIGHT_CLIN_COLS].copy()


def merge_with_light_clinical(
    X_feat: pd.DataFrame,
    tissue: str,
    combo: str,
    tp: int,
) -> pd.DataFrame:
    clin_path = build_clinical_path(tissue=tissue, combo=combo, tp=tp)
    if not clin_path.exists():
        raise FileNotFoundError(f"clinical file not found: {clin_path}")

    X_clin = load_light_clinical_csv(clin_path)

    common_idx = X_feat.index.intersection(X_clin.index)
    X_feat2 = X_feat.loc[common_idx].copy()
    X_clin2 = X_clin.loc[common_idx].copy()

    overlap_cols = [c for c in X_clin2.columns if c in X_feat2.columns]
    if len(overlap_cols) > 0:
        X_feat2 = X_feat2.drop(columns=overlap_cols)

    X_out = pd.concat([X_feat2, X_clin2], axis=1)
    return X_out


def align_xy(X: pd.DataFrame, y: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    idx = X.index.intersection(y.index)
    return X.loc[idx].copy(), y.loc[idx].copy()


In [29]:
# ==========================================
# Fold-safe sparse filter / preprocess / model
# ==========================================

def make_preprocess(X: pd.DataFrame) -> ColumnTransformer:
    cat_cols = [c for c in X.columns if c in ["Gender", "Level", "AIS"]]
    num_cols = [c for c in X.columns if c not in cat_cols]

    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), num_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore")),
            ]), cat_cols),
        ],
        remainder="drop"
    )



def make_model(model_type: str):
    if model_type == "lasso":
        return Lasso(max_iter=50000, tol=1e-3, random_state=42)
    elif model_type == "elasticnet":
        return ElasticNet(max_iter=50000, tol=1e-3, random_state=42)
    elif model_type == "ridge":
        return Ridge()
    elif model_type == "pls":
        return PLSRegression(scale=False)
    elif model_type == "svr_linear":
        return SVR(kernel="linear")
    elif model_type == "svr_rbf":
        return SVR(kernel="rbf")
    elif model_type == "extratrees":
        return ExtraTreesRegressor(
            random_state=42,
            n_jobs=1,
        )
    elif model_type == "gbr":
        return GradientBoostingRegressor(
            random_state=42,
        )
    else:
        raise ValueError(model_type)


def get_model_params(params: dict) -> dict:
    """Remove search-only parameters before passing params into the sklearn model."""
    search_only = {
        "prefilter_k",
        "weight_mode",
        "top_k",
        "feature_selection_mode",
    }
    return {k: v for k, v in params.items() if k not in search_only}


def make_param_grid(model_type: str) -> dict:
    # This legacy pipeline grid is kept for compatibility with older helper functions.
    # The paper-style weighted search uses the explicit param_space in fit_best_weighted_model().
    if model_type == "lasso":
        return {
            "model__alpha": [0.05, 0.1, 0.2, 0.5, 1.0],
            "select__k": [40, 60, 80],
        }
    elif model_type == "elasticnet":
        return {
            "model__alpha": [0.05, 0.1, 0.2, 0.5, 1.0],
            "model__l1_ratio": [0.1, 0.2, 0.4, 0.6],
            "select__k": [40, 60, 80],
        }
    elif model_type == "ridge":
        return {
            "model__alpha": [5.0, 10.0, 20.0, 50.0, 100.0],
            "select__k": [60, 80, 100],
        }
    elif model_type == "pls":
        return {
            "model__n_components": [2, 3, 5],
            "select__k": [40, 60, 80],
        }
    elif model_type in ["svr_linear", "svr_rbf"]:
        return {
            "model__C": [0.1, 1.0, 10.0],
            "model__epsilon": [0.1, 1.0],
            "select__k": [40, 60, 80],
        }
    elif model_type == "extratrees":
        return {
            "model__n_estimators": [100, 200],
            "model__max_depth": [2, 3, None],
            "model__min_samples_leaf": [2, 4],
            "select__k": [40, 60, 80],
        }
    elif model_type == "gbr":
        return {
            "model__n_estimators": [50, 100],
            "model__learning_rate": [0.03, 0.05],
            "model__max_depth": [1, 2],
            "model__min_samples_leaf": [2, 4],
            "select__k": [40, 60, 80],
        }
    else:
        raise ValueError(model_type)


def make_pipeline(X: pd.DataFrame, model_type: str) -> Pipeline:
    return Pipeline([
        ("preprocess", make_preprocess(X)),
        ("select", SelectKBest(score_func=f_regression)),
        ("model", make_model(model_type)),
    ])


In [30]:
def split_omics_blocks_transformed(columns, protected_keep_cols=None):
    """
    preprocess 이후 컬럼명 기준으로 PROT / MET / RNA / CLIN 구분.
    실제 transformed name 예:
      num__CSF_PROT_...
      num__SER_MET_...
      num__RNA_...
      num__Age
      cat__Gender_M
      cat__Level_C
    """
    if protected_keep_cols is None:
        protected_keep_cols = []

    protected_keep_cols = set(protected_keep_cols)

    prot_cols = []
    met_cols = []
    rna_cols = []
    clin_cols = []

    for c in columns:
        c_str = str(c)
        c_low = c_str.lower()

        # transformed clinical
        if any(
            
            c_low == f"num__{x.lower()}" or c_low.startswith(f"cat__{x.lower()}_")
            for x in protected_keep_cols
        ):
            clin_cols.append(c)
            continue

        # transformed omics
        if "prot_" in c_low:
            prot_cols.append(c)
        elif "met_" in c_low:
            met_cols.append(c)
        elif "rna_" in c_low:
            rna_cols.append(c)

    return {
        "PROT": prot_cols,
        "MET": met_cols,
        "RNA": rna_cols,
        "CLIN": clin_cols,
    }


In [31]:
def filter_sparse_features_train_test(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    min_obs_frac: float = 0.70,
    protected_keep_cols: list[str] | None = None,
    fallback_thresholds: list[float] | None = None,
    min_omics_features: int = 50,
):
    """
    Fold-safe sparse filter.
    1) train 기준으로 all-missing / constant column 제거
    2) train 기준 관측률 threshold 적용
    3) protected clinical columns는 강제 유지
    """
    if protected_keep_cols is None:
        protected_keep_cols = []

    if fallback_thresholds is None:
        fallback_thresholds = [min_obs_frac, 0.60, 0.50, 0.40, 0.30, 0.20]

    thresholds = sorted(set(float(x) for x in fallback_thresholds), reverse=True)

    base_cols = list(X_train.columns)
    omics_cols = [c for c in base_cols if c not in protected_keep_cols]

    # 1) train 기준 all-missing 제거
    non_all_missing_cols = X_train.columns[X_train.notna().any(axis=0)].tolist()

    # 2) train 기준 constant 제거
    constant_cols = []
    for c in non_all_missing_cols:
        s = X_train[c]
        nunq = s.dropna().nunique()
        if nunq <= 1:
            constant_cols.append(c)

    candidate_cols = [c for c in non_all_missing_cols if c not in constant_cols]

    # protected clinical은 있으면 항상 유지
    for c in protected_keep_cols:
        if c in X_train.columns and c not in candidate_cols:
            candidate_cols.append(c)

    candidate_cols = [c for c in X_train.columns if c in candidate_cols]

    chosen_thresh = None
    chosen_keep_cols = None

    X_train_base = X_train[candidate_cols].copy()
    X_valid_base = X_valid[candidate_cols].copy()

    for thresh in thresholds:
        obs_rate = X_train_base.notna().mean(axis=0)
        keep_cols = obs_rate[obs_rate >= thresh].index.tolist()

        for c in protected_keep_cols:
            if c in X_train_base.columns and c not in keep_cols:
                keep_cols.append(c)

        keep_cols = [c for c in X_train_base.columns if c in keep_cols]
        omics_keep_cols = [c for c in keep_cols if c not in protected_keep_cols]

        block_map_tmp = split_omics_blocks_transformed(
            X_train_base[keep_cols].columns,
            protected_keep_cols=protected_keep_cols,
        )

        enough_total = len(omics_keep_cols) >= min_omics_features
        enough_blocks = (
            len(block_map_tmp["PROT"]) >= 20 and
            len(block_map_tmp["MET"]) >= 20
        )

        if enough_total and enough_blocks:
            chosen_thresh = thresh
            chosen_keep_cols = keep_cols
            break
    
    if chosen_keep_cols is None:
        chosen_thresh = thresholds[-1]
        obs_rate = X_train_base.notna().mean(axis=0)
        chosen_keep_cols = obs_rate[obs_rate >= chosen_thresh].index.tolist()

        for c in protected_keep_cols:
            if c in X_train_base.columns and c not in chosen_keep_cols:
                chosen_keep_cols.append(c)

        chosen_keep_cols = [c for c in X_train_base.columns if c in chosen_keep_cols]

    X_train_f = X_train_base.loc[:, chosen_keep_cols].copy()
    X_valid_f = X_valid_base.loc[:, chosen_keep_cols].copy()

    omics_keep_cols = [c for c in chosen_keep_cols if c not in protected_keep_cols]
    protected_kept = [c for c in chosen_keep_cols if c in protected_keep_cols]

    info = {
        "chosen_thresh": float(chosen_thresh),
        "n_total_before": int(X_train.shape[1]),
        "n_total_after": int(X_train_f.shape[1]),
        "n_omics_after": int(len(omics_keep_cols)),
        "n_protected_after": int(len(protected_kept)),
        "n_all_missing_removed": int(len([c for c in base_cols if c not in non_all_missing_cols])),
        "n_constant_removed": int(len(constant_cols)),
        "kept_columns": chosen_keep_cols,
    }

    return X_train_f, X_valid_f, info

In [32]:
def drop_train_rows_with_no_omics(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    protected_keep_cols: list[str] | None = None,
):
    """
    train fold에서 omics 값이 하나도 없는 row 제거.
    valid/test는 제거하지 않음.
    """
    if protected_keep_cols is None:
        protected_keep_cols = []

    omics_cols = [c for c in X_train.columns if c not in protected_keep_cols]

    if len(omics_cols) == 0:
        info = {
            "n_train_before": int(len(X_train)),
            "n_train_after": int(len(X_train)),
            "n_removed": 0,
        }
        return X_train.copy(), y_train.copy(), info

    row_has_any_omics = X_train[omics_cols].notna().any(axis=1)
    X_out = X_train.loc[row_has_any_omics].copy()
    y_out = y_train.loc[X_out.index].copy()

    info = {
        "n_train_before": int(len(X_train)),
        "n_train_after": int(len(X_out)),
        "n_removed": int((~row_has_any_omics).sum()),
    }
    return X_out, y_out, info

In [33]:
def prepare_train_valid_fold(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_valid: pd.DataFrame,
    protected_keep_cols: list[str] | None = None,
    min_obs_frac: float = 0.70,
    fallback_thresholds: list[float] | None = None,
    min_omics_features: int = 50,
):
    """
    inner/outer 공통 fold-safe preparation
    1) train에서 omics all-missing row 제거
    2) train 기준 sparse filtering
    3) valid는 row 제거 없이 train 기준 컬럼만 맞춤
    """
    if protected_keep_cols is None:
        protected_keep_cols = []

    X_train2, y_train2, row_info = drop_train_rows_with_no_omics(
        X_train=X_train,
        y_train=y_train,
        protected_keep_cols=protected_keep_cols,
    )

    X_train_f, X_valid_f, sparse_info = filter_sparse_features_train_test(
        X_train=X_train2,
        X_valid=X_valid,
        min_obs_frac=min_obs_frac,
        protected_keep_cols=protected_keep_cols,
        fallback_thresholds=fallback_thresholds,
        min_omics_features=min_omics_features,
    )

    if X_train_f.shape[1] < MIN_FEATURES_AFTER_FILTER:
        raise ValueError(
            f"too few total features after sparse filtering: {X_train_f.shape[1]}"
        )

    if sparse_info["n_omics_after"] < min_omics_features:
        raise ValueError(
            f"too few omics features after sparse filtering: {sparse_info['n_omics_after']}"
        )

    info = {
        "row_info": row_info,
        "sparse_info": sparse_info,
    }
    return X_train_f, y_train2, X_valid_f, info

In [34]:
def build_local_feature_score_df(
    model,
    selected_features: list[str],
) -> pd.DataFrame:
    """
    한 split 안에서 선택된 feature만 가지고 local feature score 생성.
    leakage 없이 split-local weighting에 사용.
    """
    if len(selected_features) == 0:
        return pd.DataFrame(columns=["feature", "selection_freq", "mean_abs_coef", "feature_score"])

    if hasattr(model, "coef_"):
        coef_abs = np.abs(np.ravel(model.coef_))
        if len(coef_abs) != len(selected_features):
            coef_abs = np.ones(len(selected_features), dtype=float)
    elif hasattr(model, "feature_importances_"):
        coef_abs = np.abs(np.ravel(model.feature_importances_))
        if len(coef_abs) != len(selected_features):
            coef_abs = np.ones(len(selected_features), dtype=float)
    else:
        # Nonlinear models such as RBF-SVR do not expose a direct coefficient.
        # In that case the split-local feature score falls back to equal score.
        coef_abs = np.ones(len(selected_features), dtype=float)

    df = pd.DataFrame({
        "feature": selected_features,
        "selection_freq": 1.0,
        "mean_abs_coef": coef_abs,
    })

    max_coef = max(float(df["mean_abs_coef"].max()), 1e-12)
    df["feature_score"] = df["mean_abs_coef"] / max_coef
    df = df.sort_values("feature_score", ascending=False).reset_index(drop=True)
    return df

def split_omics_blocks(columns, protected_keep_cols=None):
    if protected_keep_cols is None:
        protected_keep_cols = []

    protected_keep_cols = set(protected_keep_cols)

    prot_cols = [c for c in columns if c.startswith("PROT_") and c not in protected_keep_cols]
    met_cols  = [c for c in columns if c.startswith("MET_") and c not in protected_keep_cols]
    rna_cols  = [c for c in columns if c.startswith("RNA_") and c not in protected_keep_cols]
    clin_cols = [c for c in columns if c in protected_keep_cols]

    return {
        "PROT": prot_cols,
        "MET": met_cols,
        "RNA": rna_cols,
        "CLIN": clin_cols,
    }


def allocate_blockwise_k(block_map, total_k, min_per_block=10):
    """
    total_k를 PROT/MET/RNA block에 비례 배분.
    block size가 작은 경우 가능한 범위 내에서만 배정.
    """
    omics_names = ["PROT", "MET", "RNA"]
    sizes = {k: len(block_map.get(k, [])) for k in omics_names}
    active = [k for k in omics_names if sizes[k] > 0]

    if total_k is None or len(active) == 0:
        return {k: sizes[k] for k in omics_names}

    total_k = int(total_k)
    total_available = sum(sizes[k] for k in active)
    total_k = min(total_k, total_available)

    alloc = {k: 0 for k in omics_names}

    # 1) active block에 최소 quota 부여
    for k in active:
        alloc[k] = min(min_per_block, sizes[k])

    used = sum(alloc.values())

    # total_k가 너무 작으면 최소 quota를 다시 줄임
    if used > total_k:
        alloc = {k: 0 for k in omics_names}
        base = max(1, total_k // len(active))
        rem = total_k

        for k in active:
            take = min(base, sizes[k], rem)
            alloc[k] = take
            rem -= take

        for k in sorted(active, key=lambda x: sizes[x], reverse=True):
            if rem <= 0:
                break
            extra = min(sizes[k] - alloc[k], rem)
            alloc[k] += extra
            rem -= extra

        return alloc

    # 2) 남은 quota를 block size 비례로 배분
    remaining = total_k - used
    if remaining > 0:
        size_sum = sum(sizes[k] for k in active)
        for k in active:
            extra = int(round(remaining * sizes[k] / size_sum))
            alloc[k] += extra

        # overflow / underflow 정리
        for k in active:
            alloc[k] = min(alloc[k], sizes[k])

        cur = sum(alloc.values())

        if cur < total_k:
            for k in sorted(active, key=lambda x: sizes[x] - alloc[x], reverse=True):
                if cur >= total_k:
                    break
                room = sizes[k] - alloc[k]
                take = min(room, total_k - cur)
                alloc[k] += take
                cur += take

        elif cur > total_k:
            for k in sorted(active, key=lambda x: alloc[x], reverse=True):
                if cur <= total_k:
                    break
                drop = min(alloc[k], cur - total_k)
                alloc[k] -= drop
                cur -= drop

    return alloc


In [35]:
# ==========================================
# Base model fit / OOF
# ==========================================
def fit_best_model(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    groups_train: np.ndarray,
    model_type: str,
    n_splits_inner: int = 3,
):
    n_unique_groups = int(pd.Series(groups_train).nunique())
    inner_splits = min(n_splits_inner, n_unique_groups)

    if inner_splits < 2:
        raise ValueError(
            f"Not enough unique groups for inner CV: "
            f"n_unique_groups={n_unique_groups}, requested={n_splits_inner}"
        )

    inner_cv = GroupKFold(n_splits=inner_splits)

    pipe = make_pipeline(X_train, model_type=model_type)
    grid = make_param_grid(model_type=model_type)

    gs = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring="r2",
        cv=inner_cv,
        n_jobs=8,
        refit=True,
    )
    gs.fit(X_train, y_train, groups=groups_train)
    return gs.best_estimator_, gs.best_params_, gs.best_score_

def get_inner_oof_predictions(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    groups_train: np.ndarray,
    model_type: str,
    best_params: dict,
    n_splits_inner: int = 3,
) -> np.ndarray:
    n_unique_groups = int(pd.Series(groups_train).nunique())
    inner_splits = min(n_splits_inner, n_unique_groups)

    if inner_splits < 2:
        return np.full(len(X_train), np.nan, dtype=float)

    inner_cv = GroupKFold(n_splits=inner_splits)
    oof = np.zeros(len(X_train), dtype=float)

    for tr_idx, va_idx in inner_cv.split(X_train, y_train, groups=groups_train):
        X_tr = X_train.iloc[tr_idx]
        y_tr = y_train.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]

        pipe = make_pipeline(X_tr, model_type=model_type)
        pipe.set_params(**best_params)
        pipe.fit(X_tr, y_tr)
        oof[va_idx] = pipe.predict(X_va)

    return oof


def get_selected_feature_names(fitted_pipeline: Pipeline) -> list[str]:
    preprocess = fitted_pipeline.named_steps["preprocess"]
    selector = fitted_pipeline.named_steps["select"]
    names = preprocess.get_feature_names_out()
    mask = selector.get_support()
    return list(np.array(names)[mask])

def fit_fold_imputer(X_train: pd.DataFrame, strategy: str = "median"):
    """
    Fold-safe pre-imputer.
    숫자 컬럼만 지정된 strategy로 채우고,
    object/category 컬럼은 건드리지 않는다.
    """
    num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    non_num_cols = [c for c in X_train.columns if c not in num_cols]

    imp_num = None
    if len(num_cols) > 0:
        imp_num = SimpleImputer(strategy=strategy)
        imp_num.fit(X_train[num_cols])

    return {
        "num_cols": num_cols,
        "non_num_cols": non_num_cols,
        "imp_num": imp_num,
        "strategy": strategy,
    }


def apply_fold_imputer(imputer_state, X: pd.DataFrame) -> pd.DataFrame:
    num_cols = imputer_state["num_cols"]
    non_num_cols = imputer_state["non_num_cols"]
    imp_num = imputer_state["imp_num"]

    parts = []

    if len(num_cols) > 0:
        X_num = pd.DataFrame(
            imp_num.transform(X[num_cols]),
            index=X.index,
            columns=num_cols,
        )
        parts.append(X_num)

    if len(non_num_cols) > 0:
        X_non_num = X[non_num_cols].copy()
        parts.append(X_non_num)

    if len(parts) == 0:
        return X.copy()

    X_out = pd.concat(parts, axis=1)
    X_out = X_out[X.columns]  # 원래 컬럼 순서 복원
    return X_out

In [36]:
def apply_feature_weights_to_matrix(
    X_t: pd.DataFrame,
    weight_df: pd.DataFrame,
) -> pd.DataFrame:
    w = pd.Series(1.0, index=X_t.columns, dtype=float)

    if weight_df is not None and len(weight_df) > 0:
        w_update = weight_df.set_index("feature")["weight"]
        common = X_t.columns.intersection(w_update.index)
        w.loc[common] = w_update.loc[common]

    return X_t.mul(w, axis=1)

In [37]:
def fit_preprocess_only(X_train: pd.DataFrame):
    preprocess = make_preprocess(X_train)
    preprocess.fit(X_train)
    return preprocess


def transform_with_preprocess(preprocess, X: pd.DataFrame) -> pd.DataFrame:
    
    Xt = preprocess.transform(X)
    feat_names = preprocess.get_feature_names_out().tolist()

    if sparse.issparse(Xt):
        Xt = Xt.toarray()

    Xt = np.asarray(Xt)

    if Xt.ndim == 1:
        Xt = Xt.reshape(-1, 1)

    if Xt.shape[1] != len(feat_names):
        raise ValueError(
            f"transform_with_preprocess column mismatch: "
            f"Xt.shape={Xt.shape}, len(feat_names)={len(feat_names)}"
        )

    return pd.DataFrame(Xt, index=X.index, columns=feat_names)


def allocate_blockwise_k(block_map, total_k, min_per_block=10):
    omics_names = ["PROT", "MET", "RNA"]
    sizes = {k: len(block_map.get(k, [])) for k in omics_names}
    active = [k for k in omics_names if sizes[k] > 0]

    if total_k is None or len(active) == 0:
        return {k: sizes[k] for k in omics_names}

    total_k = int(total_k)
    total_available = sum(sizes[k] for k in active)
    total_k = min(total_k, total_available)

    alloc = {k: 0 for k in omics_names}

    for k in active:
        alloc[k] = min(min_per_block, sizes[k])

    used = sum(alloc.values())

    if used > total_k:
        alloc = {k: 0 for k in omics_names}
        base = max(1, total_k // len(active))
        rem = total_k

        for k in active:
            take = min(base, sizes[k], rem)
            alloc[k] = take
            rem -= take

        for k in sorted(active, key=lambda x: sizes[x], reverse=True):
            if rem <= 0:
                break
            extra = min(sizes[k] - alloc[k], rem)
            alloc[k] += extra
            rem -= extra

        return alloc

    remaining = total_k - used
    if remaining > 0:
        size_sum = sum(sizes[k] for k in active)
        for k in active:
            extra = int(round(remaining * sizes[k] / size_sum))
            alloc[k] += extra

        for k in active:
            alloc[k] = min(alloc[k], sizes[k])

        cur = sum(alloc.values())

        if cur < total_k:
            for k in sorted(active, key=lambda x: sizes[x] - alloc[x], reverse=True):
                if cur >= total_k:
                    break
                room = sizes[k] - alloc[k]
                take = min(room, total_k - cur)
                alloc[k] += take
                cur += take

        elif cur > total_k:
            for k in sorted(active, key=lambda x: alloc[x], reverse=True):
                if cur <= total_k:
                    break
                drop = min(alloc[k], cur - total_k)
                alloc[k] -= drop
                cur -= drop

    return alloc

In [38]:

def build_feature_weights(
    feature_score_df: pd.DataFrame,
    mode: str = "none",
    top_k: int = 50,
    high_weight: float | None = None,
    low_weight: float | None = None,
    min_weight: float | None = None,
    max_weight: float | None = None,
):
    """
    Build feature-wise weights from train-side feature scores only.

    Safe default:
    - none: all weights = 1.0
    - soft: feature_score를 1.0~1.5 범위로 부드럽게 scaling
    - topk: optional, but low weight is also >=1.0 to avoid aggressively killing features

    NOTE:
    feature_score_df must be computed inside the current train fold only.
    """
    df = feature_score_df.copy()

    if len(df) == 0:
        return pd.DataFrame(columns=[
            "feature", "weight", "feature_weight_mode",
            "feature_weight_min", "feature_weight_max"
        ])

    high_weight = FEATURE_WEIGHT_TOPK_HIGH if high_weight is None else high_weight
    low_weight = FEATURE_WEIGHT_TOPK_LOW if low_weight is None else low_weight
    min_weight = FEATURE_WEIGHT_MIN if min_weight is None else min_weight
    max_weight = FEATURE_WEIGHT_MAX if max_weight is None else max_weight

    # enforce safe range.
    # v4 allows mild down-weighting (< 1.0) for low-scoring features,
    # while keeping the default unweighted value at 1.0.
    low_weight = max(float(low_weight), float(FEATURE_WEIGHT_MIN))
    min_weight = max(float(min_weight), float(FEATURE_WEIGHT_MIN))
    high_weight = min(float(high_weight), float(max_weight))
    max_weight = max(float(max_weight), float(min_weight))

    if "feature_score" not in df.columns:
        df["feature_score"] = 0.0

    if mode == "none":
        df["weight"] = float(FEATURE_WEIGHT_DEFAULT)

    elif mode == "topk":
        df["weight"] = float(low_weight)
        top_feats = df.head(int(min(top_k, len(df))))["feature"].tolist()
        df.loc[df["feature"].isin(top_feats), "weight"] = float(high_weight)

    elif mode == "soft":
        s = pd.to_numeric(df["feature_score"], errors="coerce").fillna(0.0).values.astype(float)
        s_min, s_max = np.nanmin(s), np.nanmax(s)

        if np.isclose(s_min, s_max):
            scaled = np.ones_like(s)
        else:
            scaled = (s - s_min) / (s_max - s_min)

        df["weight"] = float(min_weight) + scaled * (float(max_weight) - float(min_weight))

    else:
        raise ValueError(f"Unknown weighting mode: {mode}")

    df["weight"] = pd.to_numeric(df["weight"], errors="coerce").fillna(FEATURE_WEIGHT_DEFAULT)
    df["weight"] = df["weight"].clip(lower=FEATURE_WEIGHT_MIN, upper=FEATURE_WEIGHT_MAX)

    df["feature_weight_mode"] = mode
    df["feature_weight_min"] = FEATURE_WEIGHT_MIN
    df["feature_weight_max"] = FEATURE_WEIGHT_MAX

    return df[[
        "feature", "weight", "feature_weight_mode",
        "feature_weight_min", "feature_weight_max"
    ]].copy()


In [39]:

def compute_univariate_feature_scores(
    X_train_use: pd.DataFrame,
    y_train: pd.Series,
    feature_selection_mode: str = "f_regression_topk",
) -> pd.Series:
    """
    Fold-safe univariate score computed using the current train split only.

    Modes:
    - f_regression_topk: sklearn f_regression score, closest to paper-style univariate association
    - corr_topk: absolute Pearson correlation with DeltaTMS
    - hybrid_topk: average rank of f_regression and absolute correlation
    """
    if X_train_use.shape[1] == 0:
        return pd.Series(dtype=float)

    if feature_selection_mode == "f_regression_topk":
        scores, _ = f_regression(X_train_use, y_train)
        score_s = pd.Series(scores, index=X_train_use.columns)

    elif feature_selection_mode == "corr_topk":
        y_num = pd.to_numeric(y_train, errors="coerce")
        vals = {}
        for c in X_train_use.columns:
            x = pd.to_numeric(X_train_use[c], errors="coerce")
            common = x.notna() & y_num.notna()
            if common.sum() < 3:
                vals[c] = 0.0
            else:
                vals[c] = abs(float(np.corrcoef(x.loc[common], y_num.loc[common])[0, 1]))
        score_s = pd.Series(vals)

    elif feature_selection_mode == "hybrid_topk":
        f_scores, _ = f_regression(X_train_use, y_train)
        f_s = (
            pd.Series(f_scores, index=X_train_use.columns)
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
        )
        y_num = pd.to_numeric(y_train, errors="coerce")
        corr_vals = {}
        for c in X_train_use.columns:
            x = pd.to_numeric(X_train_use[c], errors="coerce")
            common = x.notna() & y_num.notna()
            if common.sum() < 3:
                corr_vals[c] = 0.0
            else:
                corr_vals[c] = abs(float(np.corrcoef(x.loc[common], y_num.loc[common])[0, 1]))
        c_s = pd.Series(corr_vals).replace([np.inf, -np.inf], np.nan).fillna(0.0)

        # Convert to percentile ranks so the two score scales are comparable.
        score_s = 0.5 * f_s.rank(pct=True) + 0.5 * c_s.rank(pct=True)

    else:
        raise ValueError(f"Unknown feature_selection_mode: {feature_selection_mode}")

    return (
        score_s
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .astype(float)
    )


def select_features_manual(
    X_train_t: pd.DataFrame,
    y_train: pd.Series,
    X_valid_t: pd.DataFrame,
    k: int | None = None,
    feature_selection_mode: str = "f_regression_topk",
):
    if X_train_t.shape[1] == 0:
        return X_train_t.copy(), X_valid_t.copy(), []

    # variance 0 제거
    valid_cols = X_train_t.columns[X_train_t.var(axis=0) > 0].tolist()
    if len(valid_cols) == 0:
        return (
            X_train_t.iloc[:, :0].copy(),
            X_valid_t.iloc[:, :0].copy(),
            [],
        )

    X_train_use = X_train_t[valid_cols].copy()
    X_valid_use = X_valid_t[valid_cols].copy()

    # 전체 유지
    if k is None or k >= X_train_use.shape[1]:
        return X_train_use.copy(), X_valid_use.copy(), X_train_use.columns.tolist()

    # global fallback helper
    def _global_select():
        score_s = compute_univariate_feature_scores(
            X_train_use=X_train_use,
            y_train=y_train,
            feature_selection_mode=feature_selection_mode,
        )
        selected_global = score_s.sort_values(ascending=False).head(k).index.tolist()
        return (
            X_train_use[selected_global].copy(),
            X_valid_use[selected_global].copy(),
            selected_global,
        )

    if not USE_BLOCKWISE_EF:
        return _global_select()

    block_map = split_omics_blocks_transformed(
        X_train_use.columns,
        protected_keep_cols=LIGHT_CLIN_COLS,
    )

    print(
        "[DEBUG][BLOCKWISE] counts =",
        {
            "PROT": len(block_map["PROT"]),
            "MET": len(block_map["MET"]),
            "RNA": len(block_map["RNA"]),
            "CLIN": len(block_map["CLIN"]),
        }
    )

    # block를 하나도 못 찾으면 global fallback
    n_omics_found = len(block_map["PROT"]) + len(block_map["MET"]) + len(block_map["RNA"])
    if n_omics_found == 0:
        print("[WARN][BLOCKWISE] no omics block found from transformed columns -> fallback to global selection")
        return _global_select()

    alloc = allocate_blockwise_k(
        block_map=block_map,
        total_k=k,
        min_per_block=BLOCKWISE_MIN_PER_BLOCK,
    )

    selected = []

    # clinical transformed columns 유지
    clin_cols = [c for c in block_map["CLIN"] if c in X_train_use.columns]
    selected.extend(clin_cols)

    for block_name in ["PROT", "MET", "RNA"]:
        cols = [c for c in block_map[block_name] if c in X_train_use.columns]
        if len(cols) == 0:
            continue

        block_k = alloc.get(block_name, 0)
        if block_k <= 0:
            continue

        X_tr_block = X_train_use[cols].copy()
        score_s = compute_univariate_feature_scores(
            X_train_use=X_tr_block,
            y_train=y_train,
            feature_selection_mode=feature_selection_mode,
        )

        chosen = score_s.sort_values(ascending=False).head(block_k).index.tolist()
        selected.extend(chosen)

    selected = list(dict.fromkeys(selected))
    selected = [c for c in selected if c in X_train_use.columns]

    if len(selected) == 0:
        return _global_select()

    return (
        X_train_use[selected].copy(),
        X_valid_use[selected].copy(),
        selected,
    )


In [40]:
def collect_feature_scores_inner_cv(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    groups_train: np.ndarray,
    model_type: str,
    model_params: dict,
    n_splits_inner: int = 3,
    prefilter_k: int | None = None,
    feature_selection_mode: str = "f_regression_topk",
):
    if X_train.shape[1] == 0:
        return (
            pd.DataFrame(columns=["feature", "selection_freq", "mean_abs_coef", "feature_score"]),
            pd.DataFrame(),
        )

    inner_splits_list, inner_cv_type = make_inner_cv_splits(
        X=X_train,
        y=y_train,
        groups=groups_train,
        n_splits=n_splits_inner,
    )

    inner_splits = len(inner_splits_list)

    if inner_splits < 2:
        return (
            pd.DataFrame(columns=["feature", "selection_freq", "mean_abs_coef", "feature_score"]),
            pd.DataFrame(),
        )

    freq_counter = {}
    coef_sum = {}
    coef_count = {}
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(inner_splits_list, start=1):
        X_tr_raw = X_train.iloc[tr_idx].copy()
        y_tr_raw = y_train.iloc[tr_idx].copy()
        X_va_raw = X_train.iloc[va_idx].copy()
        y_va = y_train.iloc[va_idx].copy()

        try:
            X_tr, y_tr, X_va, prep_info = prepare_train_valid_fold(
                X_train=X_tr_raw,
                y_train=y_tr_raw,
                X_valid=X_va_raw,
                protected_keep_cols=LIGHT_CLIN_COLS,
                min_obs_frac=MIN_OBS_FRAC,
                fallback_thresholds=SPARSE_FALLBACK_THRESHOLDS,
                min_omics_features=MIN_OMICS_FEATURES_AFTER_FILTER,
            )
        except Exception:
            continue

        preprocess = fit_preprocess_only(X_tr)
        X_tr_t = transform_with_preprocess(preprocess, X_tr)
        X_va_t = transform_with_preprocess(preprocess, X_va)

        X_tr_s, X_va_s, selected_feats = select_features_manual(
            X_train_t=X_tr_t,
            y_train=y_tr,
            X_valid_t=X_va_t,
            k=prefilter_k,
            feature_selection_mode=feature_selection_mode,
        )

        block_map_selected = split_omics_blocks_transformed(X_tr_s.columns, protected_keep_cols=LIGHT_CLIN_COLS)

        print(
            f"[INNER-SELECT] fold={fold} model={model_type} "
            f"| prefilter_k={prefilter_k} "
            f"| feature_selection={feature_selection_mode} "
            f"| before={X_tr_t.shape[1]} after={X_tr_s.shape[1]} "
            f"| prot={len(block_map_selected['PROT'])} "
            f"| met={len(block_map_selected['MET'])} "
            f"| rna={len(block_map_selected['RNA'])} "
            f"| clin={len(block_map_selected['CLIN'])}"
        )

        if X_tr_s.shape[1] == 0:
            continue

        model = make_model(model_type)
        model.set_params(**model_params)
        model.fit(X_tr_s, y_tr)

        pred_va = model.predict(X_va_s)
        fold_rows.append({
            "fold": fold,
            "inner_cv_type": inner_cv_type,
            "feature_selection_mode": feature_selection_mode,
            "valid_r2": r2_score(y_va, pred_va),
            "valid_mae": mean_absolute_error(y_va, pred_va),
            "n_train_after_row_filter": len(X_tr),
            "n_valid": len(X_va),
            "n_features_after_sparse": X_tr.shape[1],
            "n_selected_prefilter": X_tr_s.shape[1],
        })

        for f in selected_feats:
            freq_counter[f] = freq_counter.get(f, 0) + 1

        if hasattr(model, "coef_"):
            coef_abs = np.abs(np.ravel(model.coef_))
        elif hasattr(model, "feature_importances_"):
            coef_abs = np.abs(np.ravel(model.feature_importances_))
        else:
            coef_abs = np.ones(len(X_tr_s.columns), dtype=float)

        if len(coef_abs) == len(X_tr_s.columns):
            for f, c in zip(X_tr_s.columns, coef_abs):
                coef_sum[f] = coef_sum.get(f, 0.0) + float(c)
                coef_count[f] = coef_count.get(f, 0) + 1

    all_feats = sorted(set(list(freq_counter.keys()) + list(coef_sum.keys())))
    rows = []
    for f in all_feats:
        freq = freq_counter.get(f, 0) / inner_splits
        mean_coef = coef_sum.get(f, 0.0) / max(coef_count.get(f, 1), 1)
        rows.append({
            "feature": f,
            "selection_freq": freq,
            "mean_abs_coef": mean_coef,
        })

    score_df = pd.DataFrame(rows)
    if len(score_df) == 0:
        return (
            pd.DataFrame(columns=["feature", "selection_freq", "mean_abs_coef", "feature_score"]),
            pd.DataFrame(fold_rows),
        )

    score_df["freq_z"] = score_df["selection_freq"] / max(score_df["selection_freq"].max(), 1e-12)
    score_df["coef_z"] = score_df["mean_abs_coef"] / max(score_df["mean_abs_coef"].max(), 1e-12)
    score_df["feature_score"] = 0.6 * score_df["freq_z"] + 0.4 * score_df["coef_z"]
    score_df = score_df.sort_values("feature_score", ascending=False).reset_index(drop=True)

    return score_df, pd.DataFrame(fold_rows)


In [41]:

# ==========================================
# Nested-CV weighted model search
# ==========================================

def make_weighted_search_grid(model_type: str, tp: int | None = None, tissue: str | None = None) -> list[dict]:
    """
    v13 adaptive nested-CV grid.

    The EF space is tuned by tissue/timepoint routing; grid size still depends on signal strength:
      - strong: wider grid for promising windows, especially CSF 72/96 and serum 96
      - middle: moderate linear/PLS/SVR search
      - weak: small conservative linear search only
    """
    tp = int(tp) if tp is not None else None
    tissue = str(tissue).lower() if tissue is not None else "unknown"
    tuning_group = get_ef_tuning_group(tissue, tp) if tp is not None and tissue != "unknown" else "middle"
    is_late = tp is not None and tp >= LATE_TP_START
    fs_modes = FEATURE_SELECTION_MODES_TO_RUN
    weight_modes = FEATURE_WEIGHT_MODES

    rows = []

    if tuning_group == "strong":
        ridge_alphas = [1.0, 5.0, 10.0, 50.0, 100.0, 200.0]
        ridge_topks = [30, 40, 60]
        enet_alphas = [0.01, 0.05, 0.1, 0.5, 1.0]
        enet_l1s = [0.05, 0.1, 0.2, 0.4]
        enet_topks = [20, 30, 40]
        pls_components = [1, 2, 3]
        pls_topks = [20, 30, 40]
        svr_Cs = [0.1, 0.3, 1.0, 3.0]
        svr_eps = [0.05, 0.1, 0.2]
        svr_topks = [20, 30, 40]
        gbr_estimators = [50, 80, 120]
        gbr_lrs = [0.03, 0.05]
        gbr_depths = [1, 2]
        gbr_leafs = [3, 5]
        gbr_subsamples = [0.8, 1.0]
        gbr_topks = [20, 30]
    elif tuning_group == "middle":
        ridge_alphas = [5.0, 10.0, 50.0, 100.0, 200.0]
        ridge_topks = [30, 40, 60]
        enet_alphas = [0.05, 0.1, 0.5, 1.0]
        enet_l1s = [0.05, 0.1, 0.2]
        enet_topks = [20, 30, 40]
        pls_components = [1, 2]
        pls_topks = [20, 40]
        svr_Cs = [0.3, 1.0]
        svr_eps = [0.1, 0.2]
        svr_topks = [30, 40]
        gbr_estimators = [50, 80]
        gbr_lrs = [0.03]
        gbr_depths = [1]
        gbr_leafs = [3, 5]
        gbr_subsamples = [0.8]
        gbr_topks = [20]
    else:
        ridge_alphas = [10.0, 50.0, 100.0, 200.0]
        ridge_topks = [30, 40]
        enet_alphas = [0.1, 0.5, 1.0]
        enet_l1s = [0.05, 0.1]
        enet_topks = [20, 30]
        pls_components = [1]
        pls_topks = [20]
        svr_Cs = [0.3]
        svr_eps = [0.1]
        svr_topks = [30]
        gbr_estimators = [50]
        gbr_lrs = [0.03]
        gbr_depths = [1]
        gbr_leafs = [5]
        gbr_subsamples = [0.8]
        gbr_topks = [20]

    if model_type == "ridge":
        for alpha in ridge_alphas:
            for top_k in ridge_topks:
                for weight_mode in weight_modes:
                    for fs_mode in fs_modes:
                        rows.append({
                            "model_type": model_type,
                            "model_params": {"alpha": alpha},
                            "prefilter_k": max(top_k, 60 if not is_late else 40),
                            "top_k": top_k,
                            "weight_mode": weight_mode,
                            "feature_selection_mode": fs_mode,
                            "tuning_group": tuning_group,
                        })

    elif model_type == "elasticnet":
        for alpha in enet_alphas:
            for l1_ratio in enet_l1s:
                for top_k in enet_topks:
                    for weight_mode in weight_modes:
                        for fs_mode in fs_modes:
                            rows.append({
                                "model_type": model_type,
                                "model_params": {"alpha": alpha, "l1_ratio": l1_ratio},
                                "prefilter_k": max(top_k, 40),
                                "top_k": top_k,
                                "weight_mode": weight_mode,
                                "feature_selection_mode": fs_mode,
                                "tuning_group": tuning_group,
                            })

    elif model_type == "pls":
        for n_components in pls_components:
            for top_k in pls_topks:
                for fs_mode in fs_modes:
                    rows.append({
                        "model_type": model_type,
                        "model_params": {"n_components": n_components},
                        "prefilter_k": max(top_k, 40),
                        "top_k": top_k,
                        "weight_mode": "none",
                        "feature_selection_mode": fs_mode,
                        "tuning_group": tuning_group,
                    })

    elif model_type == "svr_linear":
        for C in svr_Cs:
            for epsilon in svr_eps:
                for top_k in svr_topks:
                    for weight_mode in weight_modes:
                        for fs_mode in fs_modes:
                            rows.append({
                                "model_type": model_type,
                                "model_params": {"C": C, "epsilon": epsilon},
                                "prefilter_k": max(top_k, 40),
                                "top_k": top_k,
                                "weight_mode": weight_mode,
                                "feature_selection_mode": fs_mode,
                                "tuning_group": tuning_group,
                            })

    elif model_type == "gbr":
        for n_estimators in gbr_estimators:
            for learning_rate in gbr_lrs:
                for max_depth in gbr_depths:
                    for min_samples_leaf in gbr_leafs:
                        for subsample in gbr_subsamples:
                            for top_k in gbr_topks:
                                for fs_mode in fs_modes:
                                    rows.append({
                                        "model_type": model_type,
                                        "model_params": {
                                            "n_estimators": n_estimators,
                                            "learning_rate": learning_rate,
                                            "max_depth": max_depth,
                                            "min_samples_leaf": min_samples_leaf,
                                            "subsample": subsample,
                                        },
                                        "prefilter_k": max(top_k, 30),
                                        "top_k": top_k,
                                        "weight_mode": "none",
                                        "feature_selection_mode": fs_mode,
                                        "tuning_group": tuning_group,
                                    })

    else:
        raise ValueError(f"Unsupported model_type for v13 nested CV grid: {model_type}")

    return rows

def _cap_model_params_for_data(model_type: str, model_params: dict, n_samples: int, n_features: int) -> dict:
    """Make model params safe for the current fold, especially PLS n_components."""
    out = dict(model_params)
    if model_type == "pls":
        max_components = max(1, min(int(n_samples) - 1, int(n_features)))
        requested = int(out.get("n_components", 2))
        out["n_components"] = int(max(1, min(requested, max_components)))
    return out


def _fit_weighted_config(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    config: dict,
):
    """
    Fit one weighted EF config on a given training split.
    Returns all fold-local objects needed for prediction and logging.
    """
    model_type = config["model_type"]
    prefilter_k = int(config["prefilter_k"])
    top_k = int(config["top_k"])
    weight_mode = config["weight_mode"]
    feature_selection_mode = config["feature_selection_mode"]

    preprocess = fit_preprocess_only(X_train)
    X_train_t = transform_with_preprocess(preprocess, X_train)

    X_train_s, _, selected_features = select_features_manual(
        X_train_t=X_train_t,
        y_train=y_train,
        X_valid_t=X_train_t,
        k=prefilter_k,
        feature_selection_mode=feature_selection_mode,
    )

    selected_features = list(dict.fromkeys(selected_features))
    if top_k is not None and len(selected_features) > top_k:
        score_tmp = compute_univariate_feature_scores(
            X_train_use=X_train_t[selected_features].copy(),
            y_train=y_train,
            feature_selection_mode=feature_selection_mode,
        )
        selected_features = score_tmp.sort_values(ascending=False).head(top_k).index.tolist()
        X_train_s = X_train_t[selected_features].copy()

    if len(selected_features) == 0:
        raise ValueError(f"No selected features for config={config}")

    score_s = compute_univariate_feature_scores(
        X_train_use=X_train_t[selected_features].copy(),
        y_train=y_train,
        feature_selection_mode=feature_selection_mode,
    )
    feature_score_df = pd.DataFrame({
        "feature": selected_features,
        "selection_freq": 1.0,
        "mean_abs_coef": score_s.reindex(selected_features).fillna(0.0).values,
        "feature_score": score_s.reindex(selected_features).fillna(0.0).values,
    })
    max_score = max(float(feature_score_df["feature_score"].max()), 1e-12)
    feature_score_df["feature_score"] = feature_score_df["feature_score"] / max_score
    feature_score_df = feature_score_df.sort_values("feature_score", ascending=False).reset_index(drop=True)

    weight_df = build_feature_weights(
        feature_score_df=feature_score_df,
        mode=weight_mode,
        top_k=top_k,
        min_weight=FEATURE_WEIGHT_MIN,
        max_weight=FEATURE_WEIGHT_MAX,
    )

    X_train_w = apply_feature_weights_to_matrix(X_train_s, weight_df)

    model_params = _cap_model_params_for_data(
        model_type=model_type,
        model_params=config["model_params"],
        n_samples=X_train_w.shape[0],
        n_features=X_train_w.shape[1],
    )
    model = make_model(model_type)
    model.set_params(**model_params)
    model.fit(X_train_w, y_train)

    return {
        "preprocess": preprocess,
        "selected_features": selected_features,
        "feature_score_df": feature_score_df,
        "feature_weight_df": weight_df,
        "fitted_model": model,
        "model_params": model_params,
        "X_train_w": X_train_w,
    }


def _predict_weighted_fit(fit_obj: dict, X: pd.DataFrame) -> pd.Series:
    preprocess = fit_obj["preprocess"]
    selected_features = fit_obj["selected_features"]
    weight_df = fit_obj["feature_weight_df"]
    model = fit_obj["fitted_model"]

    X_t = transform_with_preprocess(preprocess, X)
    X_s = X_t[selected_features].copy()
    w_sub = weight_df[weight_df["feature"].isin(selected_features)].copy()
    X_w = apply_feature_weights_to_matrix(X_s, w_sub)
    return pd.Series(model.predict(X_w), index=X.index, dtype=float)


def fit_best_weighted_model(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    groups_train: np.ndarray,
    candidate_models: list[str],
    n_splits_inner: int = 3,
    tp: int | None = None,
    tissue: str | None = None,
):
    """
    v10 full nested-CV weighted model search.

    Outer fold is handled by run_early_fusion_cv(). Inside the current outer training fold,
    this function runs GroupKFold inner CV to select the best config for the requested model.

    Since run_one_ef_experiment passes one model_type per job, candidate_models normally has length 1.
    Keeping the list interface preserves compatibility with older notebook code.
    """
    if len(candidate_models) < 1:
        raise ValueError("candidate_models must contain at least one model type")

    inner_splits_list, inner_cv_type = make_inner_cv_splits(
        X=X_train,
        y=y_train,
        groups=groups_train,
        n_splits=n_splits_inner,
    )
    inner_splits = len(inner_splits_list)
    if inner_splits < 2:
        n_unique_groups = int(pd.Series(groups_train).nunique())
        raise ValueError(
            f"Not enough data for inner CV: n_unique_groups={n_unique_groups}, requested={n_splits_inner}"
        )

    search_rows = []
    best_record = None

    for model_type in candidate_models:
        for cfg_idx, config in enumerate(make_weighted_search_grid(model_type=model_type, tp=tp, tissue=tissue), start=1):
            fold_scores = []
            fold_maes = []
            fold_train_scores = []
            fold_status = []

            for inner_fold, (tr_idx, va_idx) in enumerate(inner_splits_list, start=1):
                X_in_tr = X_train.iloc[tr_idx].copy()
                y_in_tr = y_train.iloc[tr_idx].copy()
                X_in_va = X_train.iloc[va_idx].copy()
                y_in_va = y_train.iloc[va_idx].copy()

                try:
                    fit_obj = _fit_weighted_config(X_in_tr, y_in_tr, config)
                    pred_va = _predict_weighted_fit(fit_obj, X_in_va)
                    pred_tr = _predict_weighted_fit(fit_obj, X_in_tr)

                    valid_r2 = r2_score(y_in_va, pred_va) if len(y_in_va) > 1 else np.nan
                    valid_mae = mean_absolute_error(y_in_va, pred_va) if len(y_in_va) > 0 else np.nan
                    train_r2 = r2_score(y_in_tr, pred_tr) if len(y_in_tr) > 1 else np.nan

                    fold_scores.append(valid_r2)
                    fold_maes.append(valid_mae)
                    fold_train_scores.append(train_r2)
                    fold_status.append("ok")
                except Exception as e:
                    fold_scores.append(np.nan)
                    fold_maes.append(np.nan)
                    fold_train_scores.append(np.nan)
                    fold_status.append(f"failed: {type(e).__name__}: {e}")

            score_arr = np.asarray(fold_scores, dtype=float)
            mae_arr = np.asarray(fold_maes, dtype=float)
            train_arr = np.asarray(fold_train_scores, dtype=float)
            ok_mask = np.isfinite(score_arr)

            mean_inner_r2 = float(np.nanmean(score_arr)) if ok_mask.any() else np.nan
            median_inner_r2 = float(np.nanmedian(score_arr)) if ok_mask.any() else np.nan
            std_inner_r2 = float(np.nanstd(score_arr, ddof=1)) if ok_mask.sum() > 1 else 0.0
            min_inner_r2 = float(np.nanmin(score_arr)) if ok_mask.any() else np.nan
            mean_inner_mae = float(np.nanmean(mae_arr)) if np.isfinite(mae_arr).any() else np.nan
            mean_train_r2 = float(np.nanmean(train_arr)) if np.isfinite(train_arr).any() else np.nan

            # Stability-aware objective. This avoids selecting configs that win only by one lucky fold.
            # v10 slightly strengthens the worst-fold term because v9 had high train R2 but weak OOF R2.
            selection_score = mean_inner_r2 - 0.50 * std_inner_r2 + 0.20 * min_inner_r2
            if not np.isfinite(selection_score):
                selection_score = -np.inf

            row = {
                "cv_search_mode": CV_SEARCH_MODE,
                "inner_cv_type": inner_cv_type,
                "tuning_group": config.get("tuning_group", ""),
                "model_type": model_type,
                "config_idx": cfg_idx,
                "tp": tp,
                "selection_score": selection_score,
                "mean_inner_r2": mean_inner_r2,
                "median_inner_r2": median_inner_r2,
                "std_inner_r2": std_inner_r2,
                "min_inner_r2": min_inner_r2,
                "mean_inner_mae": mean_inner_mae,
                "mean_inner_train_r2": mean_train_r2,
                "n_ok_inner_folds": int(ok_mask.sum()),
                "n_inner_folds": int(inner_splits),
                "prefilter_k": config["prefilter_k"],
                "top_k": config["top_k"],
                "weight_mode": config["weight_mode"],
                "feature_selection_mode": config["feature_selection_mode"],
                **{f"model_param__{k}": v for k, v in config["model_params"].items()},
            }
            for j, val in enumerate(score_arr, start=1):
                row[f"inner_fold{j}_r2"] = val
            for j, val in enumerate(mae_arr, start=1):
                row[f"inner_fold{j}_mae"] = val
            search_rows.append(row)

            if best_record is None or selection_score > best_record["selection_score"]:
                best_record = {
                    "selection_score": selection_score,
                    "config": config,
                    "row": row,
                }

    search_df = pd.DataFrame(search_rows)
    if best_record is None or not np.isfinite(best_record["selection_score"]):
        raise ValueError("No valid inner-CV config was found")

    best_config = best_record["config"]

    # Refit the selected config on the full current outer-training fold.
    final_fit = _fit_weighted_config(X_train, y_train, best_config)
    pred_train = _predict_weighted_fit(final_fit, X_train)
    train_r2 = r2_score(y_train, pred_train) if len(y_train) > 1 else np.nan
    train_mae = mean_absolute_error(y_train, pred_train) if len(y_train) > 0 else np.nan

    selected_features = final_fit["selected_features"]
    feature_score_df = final_fit["feature_score_df"].copy()

    # Add model coefficient/importances if available for interpretability.
    coef_score_df = build_local_feature_score_df(final_fit["fitted_model"], selected_features)
    if len(coef_score_df) > 0:
        feature_score_df = feature_score_df.merge(
            coef_score_df[["feature", "mean_abs_coef"]].rename(columns={"mean_abs_coef": "model_abs_coef"}),
            on="feature",
            how="left",
        )

    weight_df = final_fit["feature_weight_df"].copy()
    best_params = {
        "model_params": final_fit["model_params"],
        "prefilter_k": best_config["prefilter_k"],
        "top_k": best_config["top_k"],
        "weight_mode": best_config["weight_mode"],
        "feature_selection_mode": best_config["feature_selection_mode"],
        **{f"model_param__{k}": v for k, v in final_fit["model_params"].items()},
    }

    return {
        "best_model_type": best_config["model_type"],
        "best_params": best_params,
        "best_score": float(best_record["row"]["selection_score"]),
        "best_mean_inner_r2": float(best_record["row"]["mean_inner_r2"]),
        "best_std_inner_r2": float(best_record["row"]["std_inner_r2"]),
        "best_median_inner_r2": float(best_record["row"]["median_inner_r2"]),
        "best_min_inner_r2": float(best_record["row"]["min_inner_r2"]),
        "best_train_r2_outer_fold": train_r2,
        "best_train_mae_outer_fold": train_mae,
        "inner_cv_type": inner_cv_type,
        "search_df": search_df,
        "feature_score_df": feature_score_df,
        "feature_weight_df": weight_df,
        "preprocess": final_fit["preprocess"],
        "selected_features": selected_features,
        "fitted_model": final_fit["fitted_model"],
        "feature_weight_mode": best_config["weight_mode"],
        "feature_selection_mode": best_config["feature_selection_mode"],
        "mean_feature_weight": float(weight_df["weight"].mean()) if len(weight_df) > 0 else np.nan,
        "max_feature_weight": float(weight_df["weight"].max()) if len(weight_df) > 0 else np.nan,
        "n_weighted_features": int((weight_df["weight"] != FEATURE_WEIGHT_DEFAULT).sum()) if len(weight_df) > 0 else 0,
    }


In [42]:
def get_inner_oof_predictions_weighted(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    groups_train: np.ndarray,
    best_model_type: str,
    best_params: dict,
    n_splits_inner: int = 3,
) -> np.ndarray:
    """
    weighted feature selection/search result를 기준으로
    train-fold 내부 OOF prediction을 만든다.
    """
    inner_splits_list, inner_cv_type = make_inner_cv_splits(
        X=X_train,
        y=y_train,
        groups=groups_train,
        n_splits=n_splits_inner,
    )

    oof = pd.Series(index=X_train.index, dtype=float)

    if len(inner_splits_list) < 2:
        return np.full(len(X_train), np.nan, dtype=float)

    model_params = get_model_params(best_params)
    prefilter_k = best_params.get("prefilter_k", None)
    weight_mode = best_params.get("weight_mode", "soft")
    top_k = best_params.get("top_k", 50)
    feature_selection_mode = best_params.get("feature_selection_mode", "f_regression_topk")

    for tr_idx, va_idx in inner_splits_list:
        X_tr = X_train.iloc[tr_idx].copy()
        y_tr = y_train.iloc[tr_idx].copy()
        X_va = X_train.iloc[va_idx].copy()

        groups_tr = X_tr.index.to_numpy()

        # inner-train 안에서만 feature scoring
        score_df, _ = collect_feature_scores_inner_cv(
            X_train=X_tr,
            y_train=y_tr,
            groups_train=groups_tr,
            model_type=best_model_type,
            model_params=model_params,
            n_splits_inner=n_splits_inner,
            prefilter_k=prefilter_k,
            feature_selection_mode=feature_selection_mode,
        )

        weight_df = build_feature_weights(
            score_df,
            mode=weight_mode,
            top_k=top_k,
        )

        preprocess = fit_preprocess_only(X_tr)
        X_tr_t = transform_with_preprocess(preprocess, X_tr)
        X_va_t = transform_with_preprocess(preprocess, X_va)

        selected = score_df.head(prefilter_k)["feature"].tolist() if prefilter_k is not None else X_tr_t.columns.tolist()
        selected = [c for c in selected if c in X_tr_t.columns]

        X_tr_s = X_tr_t[selected].copy()
        X_va_s = X_va_t[selected].copy()

        w_sub = weight_df[weight_df["feature"].isin(selected)].copy()

        X_tr_w = apply_feature_weights_to_matrix(X_tr_s, w_sub)
        X_va_w = apply_feature_weights_to_matrix(X_va_s, w_sub)

        model = make_model(best_model_type)
        model.set_params(**model_params)
        model.fit(X_tr_w, y_tr)

        oof.iloc[va_idx] = model.predict(X_va_w)

    return oof.values


In [43]:
def make_regression_strata(
    y: pd.Series,
    n_bins: int,
    n_splits: int,
) -> pd.Series:
    """
    Quantile-based strata for regression CV.

    목적:
    - small-n regression에서 fold별 DeltaTMS 분포가 크게 달라지는 문제를 줄임
    - LF / single-omics와 같은 StratifiedKFold_y_bins 기준으로 EF를 비교 가능하게 만듦
    """
    y_num = pd.to_numeric(y, errors="coerce")

    if y_num.notna().sum() < n_splits:
        return pd.Series(index=y.index, data=0).astype(int)

    max_bins = min(n_bins, int(y_num.nunique()))

    for bins in range(max_bins, 1, -1):
        try:
            y_bin = pd.qcut(
                y_num,
                q=bins,
                labels=False,
                duplicates="drop",
            )
            y_bin = pd.Series(y_bin, index=y.index)

            counts = y_bin.value_counts(dropna=True)

            if len(counts) >= 2 and counts.min() >= n_splits:
                return y_bin.fillna(-1).astype(int)

        except Exception:
            continue

    return pd.Series(index=y.index, data=0).astype(int)


def make_outer_cv_splits(
    X: pd.DataFrame,
    y: pd.Series,
    groups: np.ndarray,
    n_splits: int,
):
    """
    Outer CV split helper.

    EF도 LF/single-omics와 공정하게 비교하기 위해
    기본적으로 DeltaTMS-stratified outer CV를 사용한다.

    fallback:
    - stratification이 불가능하면 GroupKFold 사용
    """
    if USE_STRATIFIED_OUTER_CV and X.index.is_unique:
        y_bins = make_regression_strata(
            y=y,
            n_bins=N_Y_BINS_FOR_OUTER_CV,
            n_splits=n_splits,
        )

        if y_bins.nunique(dropna=True) >= 2:
            outer_cv = StratifiedKFold(
                n_splits=n_splits,
                shuffle=True,
                random_state=OUTER_CV_RANDOM_STATE,
            )
            return list(outer_cv.split(X, y_bins)), "StratifiedKFold_y_bins"

    outer_cv = GroupKFold(n_splits=n_splits)
    return list(outer_cv.split(X, y, groups=groups)), "GroupKFold_fallback"

In [44]:
def make_inner_cv_splits(
    X: pd.DataFrame,
    y: pd.Series,
    groups: np.ndarray,
    n_splits: int,
):
    """
    Inner CV split helper.

    목적:
    - small-n regression에서 inner fold별 DeltaTMS 분포 차이를 줄임
    - hyperparameter / prefilter_k / weight_mode 선택을 더 안정화
    - patient-level index가 unique하면 y-bin stratified CV 사용
    - 불가능하면 GroupKFold로 fallback
    """
    n_unique_groups = int(pd.Series(groups).nunique())
    inner_splits = min(n_splits, n_unique_groups, len(X))

    if inner_splits < 2:
        return [], "too_few_samples"

    if USE_STRATIFIED_INNER_CV and X.index.is_unique:
        y_bins = make_regression_strata(
            y=y,
            n_bins=N_Y_BINS_FOR_OUTER_CV,
            n_splits=inner_splits,
        )

        if y_bins.nunique(dropna=True) >= 2:
            inner_cv = StratifiedKFold(
                n_splits=inner_splits,
                shuffle=True,
                random_state=OUTER_CV_RANDOM_STATE,
            )
            return list(inner_cv.split(X, y_bins)), "StratifiedKFold_y_bins"

    inner_cv = GroupKFold(n_splits=inner_splits)
    return list(inner_cv.split(X, y, groups=groups)), "GroupKFold_fallback"

In [45]:
def run_early_fusion_cv(
    tissue: str,
    combo: str,
    tp: int,
    y_all: pd.Series,
    model_type: str,
    clinical_mode: str = "light",
    allowed_ids: list[str] | None = None,
):
    path = build_feature_path(
        tissue=tissue,
        combo=combo,
        tp=tp,
        use_delta=USE_DELTA_VIEW,
    )

    X = load_feature_csv(path)

    if clinical_mode == "light":
        X = merge_with_light_clinical(
            X_feat=X,
            tissue=tissue,
            combo=combo,
            tp=tp,
        )
    elif clinical_mode == "omics_only":
        X = X.copy()
    else:
        raise ValueError(f"Unknown clinical_mode: {clinical_mode}")

    X, y = align_xy(X, y_all)

    # Restrict EF CV to predefined train patients only.
    # This keeps EF comparable to LF/single-omics train-only CV.
    if allowed_ids is not None:
        allowed_ids = [str(pid) for pid in allowed_ids]
        keep_ids = [pid for pid in allowed_ids if pid in X.index]
        X = X.loc[keep_ids].copy()
        y = y.loc[keep_ids].copy()

    if len(X) < N_SPLITS_OUTER:
        raise ValueError(
            f"Not enough samples after allowed_ids filtering: "
            f"n={len(X)}, n_splits={N_SPLITS_OUTER}"
        )

    groups = X.index.to_numpy()

    outer_splits, outer_cv_type = make_outer_cv_splits(
        X=X,
        y=y,
        groups=groups,
        n_splits=N_SPLITS_OUTER,
    )

    pred_oof = pd.Series(index=X.index, dtype=float)
    fold_rows = []
    feature_rows = []
    feature_score_rows = []
    feature_weight_rows = []
    search_rows_all = []

    for fold, (tr_idx, va_idx) in enumerate(outer_splits, start=1):
        X_tr = X.iloc[tr_idx].copy()
        y_tr = y.iloc[tr_idx].copy()
        X_va = X.iloc[va_idx].copy()
        y_va = y.iloc[va_idx].copy()

        # validation label distribution diagnostic
        # 모델/feature/weight 선택에는 사용하지 않고, fold R2 해석용으로만 저장
        y_va_numeric = pd.to_numeric(y_va, errors="coerce").dropna()
        valid_y_mean = float(y_va_numeric.mean()) if len(y_va_numeric) > 0 else np.nan
        valid_y_std = float(y_va_numeric.std(ddof=1)) if len(y_va_numeric) > 1 else np.nan
        valid_y_range = (
            float(y_va_numeric.max() - y_va_numeric.min())
            if len(y_va_numeric) > 0
            else np.nan
        )
        valid_y_sst = (
            float(((y_va_numeric - y_va_numeric.mean()) ** 2).sum())
            if len(y_va_numeric) > 0
            else np.nan
        )

        X_tr, y_tr, row_info = drop_train_rows_with_no_omics(
            X_train=X_tr,
            y_train=y_tr,
            protected_keep_cols=LIGHT_CLIN_COLS,
        )

        print(
            f"[ROW-FILTER][EF] tissue={tissue} combo={combo} tp={tp} clinical={clinical_mode} fold={fold} "
            f"| train_before={row_info['n_train_before']} "
            f"| train_after={row_info['n_train_after']} "
            f"| removed_no_omics={row_info['n_removed']}"
        )

        X_tr_f, X_va_f, sparse_info = filter_sparse_features_train_test(
            X_tr,
            X_va,
            min_obs_frac=MIN_OBS_FRAC,
            protected_keep_cols=LIGHT_CLIN_COLS,
            fallback_thresholds=SPARSE_FALLBACK_THRESHOLDS,
            min_omics_features=MIN_OMICS_FEATURES_AFTER_FILTER,
        )

        print(
            f"[SPARSE][EF] tissue={tissue} combo={combo} tp={tp} clinical={clinical_mode} fold={fold} "
            f"| chosen_thresh={sparse_info['chosen_thresh']:.2f} "
            f"| total_before={sparse_info['n_total_before']} "
            f"| total_after={sparse_info['n_total_after']} "
            f"| omics_after={sparse_info['n_omics_after']} "
            f"| protected_after={sparse_info['n_protected_after']} "
            f"| removed_all_missing={sparse_info['n_all_missing_removed']} "
            f"| removed_constant={sparse_info['n_constant_removed']}"
        )
        if X_tr_f.shape[1] < MIN_FEATURES_AFTER_FILTER:
            raise ValueError(
                f"[EF-{tissue}-{combo}-TP{tp}-fold{fold}] "
                f"too few total features after sparse filtering: {X_tr_f.shape[1]}"
            )

        if sparse_info["n_omics_after"] < MIN_OMICS_FEATURES_AFTER_FILTER:
            raise ValueError(
                f"too few omics features after sparse filtering: "
                f"{sparse_info['n_omics_after']} < {MIN_OMICS_FEATURES_AFTER_FILTER}"
            )




        print(
            f"[IMPUTE][EF] tissue={tissue} combo={combo} tp={tp} clinical={clinical_mode} fold={fold} "
            f"| train_shape={X_tr_f.shape} valid_shape={X_va_f.shape}"
        )

        groups_tr = X_tr_f.index.to_numpy()

        search_result = fit_best_weighted_model(
            X_train=X_tr_f,
            y_train=y_tr,
            groups_train=groups_tr,
            candidate_models=[model_type],
            n_splits_inner=N_SPLITS_INNER,
            tp=tp,
            tissue=tissue,
        )

        sel_block_map = split_omics_blocks(search_result["selected_features"], protected_keep_cols=LIGHT_CLIN_COLS)

        print(
            f"[SEARCH-RESULT][EF] tissue={tissue} combo={combo} tp={tp} clinical={clinical_mode} fold={fold} "
            f"| best_model={search_result['best_model_type']} "
            f"| selection_score={search_result['best_score']:.4f} "
            f"| mean_inner_r2={search_result.get('best_mean_inner_r2', np.nan):.4f} "
            f"| std_inner_r2={search_result.get('best_std_inner_r2', np.nan):.4f} "
            f"| best_prefilter_k={search_result['best_params']['prefilter_k']} "
            f"| weight_mode={search_result.get('feature_weight_mode', 'unknown')} "
            f"| n_selected={len(search_result['selected_features'])} "
            f"| prot={len(sel_block_map['PROT'])} "
            f"| met={len(sel_block_map['MET'])} "
            f"| rna={len(sel_block_map['RNA'])} "
            f"| clin={len(sel_block_map['CLIN'])}"
        )

        preprocess = search_result["preprocess"]
        selected_features = search_result["selected_features"]
        weight_df = search_result["feature_weight_df"]
        best_model = search_result["fitted_model"]

        print(
            f"[OUTER-FINAL] tissue={tissue} combo={combo} tp={tp} clinical={clinical_mode} fold={fold} "
            f"| fixed_model={model_type} "
            f"| selected_final={len(selected_features)} "
            f"| feature_weight_mode={search_result.get('feature_weight_mode', 'unknown')} "
            f"| mean_feature_weight={search_result.get('mean_feature_weight', np.nan):.3f} "
            f"| selection_score={search_result.get('best_score', np.nan):.4f} "
            f"| mean_inner_r2={search_result.get('best_mean_inner_r2', np.nan):.4f} "
            f"| std_inner_r2={search_result.get('best_std_inner_r2', np.nan):.4f}"
        )

        X_tr_t = transform_with_preprocess(preprocess, X_tr_f)
        X_va_t = transform_with_preprocess(preprocess, X_va_f)

        X_tr_s = X_tr_t[selected_features].copy()
        X_va_s = X_va_t[selected_features].copy()

        w_sub = weight_df[weight_df["feature"].isin(selected_features)].copy()

        X_tr_w = apply_feature_weights_to_matrix(X_tr_s, w_sub)
        X_va_w = apply_feature_weights_to_matrix(X_va_s, w_sub)

        pred_va = pd.Series(best_model.predict(X_va_w), index=X_va_w.index, dtype=float)
        pred_tr = pd.Series(best_model.predict(X_tr_w), index=X_tr_w.index, dtype=float)

        pred_oof.loc[pred_va.index] = pred_va

        
        fold_rows.append({
            "fold": fold,
            "status": "ok",
            "fusion_type": "EF",
            "experiment_tag": EXPERIMENT_TAG,
            "outer_cv_type": outer_cv_type,
            "model_type": model_type,
            "clinical_mode": clinical_mode,
            "tissue": tissue,
            "combo": combo,
            "tp": tp,

            "n_train": len(X_tr_w),
            "n_valid": len(X_va_w),
            "train_r2": r2_score(y_tr, pred_tr) if len(y_tr) > 1 else np.nan,
            "train_mae": mean_absolute_error(y_tr, pred_tr) if len(y_tr) > 0 else np.nan,
            "valid_r2": r2_score(y_va, pred_va) if len(y_va) > 1 else np.nan,
            "valid_mae": mean_absolute_error(y_va, pred_va) if len(y_va) > 0 else np.nan,

            "valid_y_mean": valid_y_mean,
            "valid_y_std": valid_y_std,
            "valid_y_range": valid_y_range,
            "valid_y_sst": valid_y_sst,

            "fixed_model_type": model_type,
            "inner_selection_score": search_result.get("best_score", np.nan),
            "inner_best_r2": search_result.get("best_mean_inner_r2", np.nan),
            "inner_r2_std": search_result.get("best_std_inner_r2", np.nan),
            "inner_r2_median": search_result.get("best_median_inner_r2", np.nan),
            "inner_r2_min": search_result.get("best_min_inner_r2", np.nan),
            "inner_cv_type": search_result.get("inner_cv_type", ""),
            "sparse_chosen_thresh": sparse_info["chosen_thresh"],
            "n_features_after_sparse": sparse_info["n_total_after"],
            "n_omics_after_sparse": sparse_info["n_omics_after"],
            "n_protected_after_sparse": sparse_info["n_protected_after"],
            "n_selected_final": len(selected_features),

            "use_feature_weighting": USE_FEATURE_WEIGHTING,
            "feature_weight_modes": ",".join(FEATURE_WEIGHT_MODES),
            "feature_weight_mode": search_result.get("feature_weight_mode", "unknown"),
            "feature_selection_mode": search_result.get("feature_selection_mode", ""),
            "feature_weight_min": FEATURE_WEIGHT_MIN,
            "feature_weight_max": FEATURE_WEIGHT_MAX,
            "mean_feature_weight": search_result.get("mean_feature_weight", np.nan),
            "max_feature_weight": search_result.get("max_feature_weight", np.nan),
            "n_weighted_features": search_result.get("n_weighted_features", np.nan),
        })

        if len(selected_features) > 0:
            tmp_feat = pd.DataFrame({"selected_feature": selected_features})
            tmp_feat["feature_rank"] = range(1, len(tmp_feat) + 1)
            tmp_feat["fold"] = fold
            tmp_feat["fusion_type"] = "EF"
            tmp_feat["experiment_tag"] = EXPERIMENT_TAG
            tmp_feat["outer_cv_type"] = outer_cv_type
            tmp_feat["model_type"] = model_type
            tmp_feat["clinical_mode"] = clinical_mode
            tmp_feat["tissue"] = tissue
            tmp_feat["combo"] = combo
            tmp_feat["tp"] = tp
            tmp_feat["use_feature_weighting"] = USE_FEATURE_WEIGHTING
            tmp_feat["feature_weight_mode"] = search_result.get("feature_weight_mode", "unknown")
            tmp_feat["feature_selection_mode"] = search_result.get("feature_selection_mode", "")
            feature_rows.append(tmp_feat)

        if isinstance(search_result.get("feature_score_df", None), pd.DataFrame):
            tmp = search_result["feature_score_df"].copy()
            tmp["fold"] = fold
            tmp["fusion_type"] = "EF"
            tmp["experiment_tag"] = EXPERIMENT_TAG
            tmp["outer_cv_type"] = outer_cv_type
            tmp["model_type"] = model_type
            tmp["clinical_mode"] = clinical_mode
            tmp["tissue"] = tissue
            tmp["combo"] = combo
            tmp["tp"] = tp
            feature_score_rows.append(tmp)

        if isinstance(weight_df, pd.DataFrame) and len(weight_df) > 0:
            tmp = weight_df.copy()
            tmp["fold"] = fold
            tmp["fusion_type"] = "EF"
            tmp["experiment_tag"] = EXPERIMENT_TAG
            tmp["outer_cv_type"] = outer_cv_type
            tmp["model_type"] = model_type
            tmp["clinical_mode"] = clinical_mode
            tmp["tissue"] = tissue
            tmp["combo"] = combo
            tmp["tp"] = tp
            tmp["use_feature_weighting"] = USE_FEATURE_WEIGHTING
            tmp["feature_weight_modes"] = ",".join(FEATURE_WEIGHT_MODES)
            tmp["feature_selection_mode"] = search_result.get("feature_selection_mode", "")
            feature_weight_rows.append(tmp)

        if isinstance(search_result.get("search_df", None), pd.DataFrame):
            tmp = search_result["search_df"].copy()
            tmp["fold"] = fold
            tmp["fusion_type"] = "EF"
            tmp["experiment_tag"] = EXPERIMENT_TAG
            tmp["outer_cv_type"] = outer_cv_type
            tmp["model_type"] = model_type
            tmp["clinical_mode"] = clinical_mode
            tmp["tissue"] = tissue
            tmp["combo"] = combo
            tmp["tp"] = tp
            search_rows_all.append(tmp)

    pred_df = pd.DataFrame({
        "Patient": pred_oof.index,
        "y_true": y.loc[pred_oof.index].values,
        "y_pred": pred_oof.values,
    }).dropna(subset=["y_pred"])

    pred_df["fusion_type"] = "EF"
    pred_df["experiment_tag"] = EXPERIMENT_TAG
    pred_df["outer_cv_type"] = outer_cv_type
    pred_df["model_type"] = model_type
    pred_df["clinical_mode"] = clinical_mode
    pred_df["tissue"] = tissue
    pred_df["combo"] = combo
    pred_df["tp"] = tp

    fold_df_tmp = pd.DataFrame(fold_rows)
    ok_fold_df = fold_df_tmp[fold_df_tmp["status"] == "ok"].copy() if len(fold_df_tmp) > 0 else pd.DataFrame()

    metrics = {
        "status": "ok" if len(ok_fold_df) > 0 else "failed",
        "fusion_type": "EF",
        "experiment_tag": EXPERIMENT_TAG,
        "outer_cv_type": outer_cv_type,
        "use_feature_weighting": USE_FEATURE_WEIGHTING,
        "feature_weight_modes": ",".join(FEATURE_WEIGHT_MODES),
        "feature_weight_min": FEATURE_WEIGHT_MIN,
        "feature_weight_max": FEATURE_WEIGHT_MAX,
        "model_type": model_type,
        "clinical_mode": clinical_mode,
        "tissue": tissue,
        "combo": combo,
        "tp": tp,

        "oof_r2": r2_score(pred_df["y_true"], pred_df["y_pred"]) if len(pred_df) > 1 else np.nan,
        "oof_mae": mean_absolute_error(pred_df["y_true"], pred_df["y_pred"]) if len(pred_df) > 0 else np.nan,
        "cv_r2": r2_score(pred_df["y_true"], pred_df["y_pred"]) if len(pred_df) > 1 else np.nan,
        "cv_mae": mean_absolute_error(pred_df["y_true"], pred_df["y_pred"]) if len(pred_df) > 0 else np.nan,

        "n_total_samples": int(len(X)),
        "n_pred_samples": int(len(pred_df)),
        "pred_coverage": float(len(pred_df) / len(X)) if len(X) > 0 else np.nan,

        "n_total_folds": int(len(fold_df_tmp)),
        "n_completed_folds": int((fold_df_tmp["status"] == "ok").sum()) if "status" in fold_df_tmp.columns else len(fold_df_tmp),
        "n_failed_folds": int((fold_df_tmp["status"] == "failed").sum()) if "status" in fold_df_tmp.columns else 0,

        "mean_valid_r2": (
            float(ok_fold_df["valid_r2"].mean())
            if len(ok_fold_df) > 0 and "valid_r2" in ok_fold_df.columns
            else np.nan
        ),
        "median_valid_r2": (
            float(ok_fold_df["valid_r2"].median())
            if len(ok_fold_df) > 0 and "valid_r2" in ok_fold_df.columns
            else np.nan
        ),
        "min_valid_r2": (
            float(ok_fold_df["valid_r2"].min())
            if len(ok_fold_df) > 0 and "valid_r2" in ok_fold_df.columns
            else np.nan
        ),
        "valid_r2_q25": (
            float(ok_fold_df["valid_r2"].quantile(0.25))
            if len(ok_fold_df) > 0 and "valid_r2" in ok_fold_df.columns
            else np.nan
        ),
        "valid_r2_q75": (
            float(ok_fold_df["valid_r2"].quantile(0.75))
            if len(ok_fold_df) > 0 and "valid_r2" in ok_fold_df.columns
            else np.nan
        ),

        "mean_valid_mae": (
            float(ok_fold_df["valid_mae"].mean())
            if len(ok_fold_df) > 0 and "valid_mae" in ok_fold_df.columns
            else np.nan
        ),
        "mean_train_r2": (
            float(ok_fold_df["train_r2"].mean())
            if len(ok_fold_df) > 0 and "train_r2" in ok_fold_df.columns
            else np.nan
        ),
        "mean_train_mae": (
            float(ok_fold_df["train_mae"].mean())
            if len(ok_fold_df) > 0 and "train_mae" in ok_fold_df.columns
            else np.nan
        ),
        "mean_inner_cv_r2": (
            float(ok_fold_df["inner_best_r2"].mean())
            if len(ok_fold_df) > 0 and "inner_best_r2" in ok_fold_df.columns
            else np.nan
        ),

        "mean_feature_weight": (
            float(ok_fold_df["mean_feature_weight"].mean())
            if len(ok_fold_df) > 0 and "mean_feature_weight" in ok_fold_df.columns
            else np.nan
        ),
        "mean_max_feature_weight": (
            float(ok_fold_df["max_feature_weight"].mean())
            if len(ok_fold_df) > 0 and "max_feature_weight" in ok_fold_df.columns
            else np.nan
        ),
        "mean_n_weighted_features": (
            float(ok_fold_df["n_weighted_features"].mean())
            if len(ok_fold_df) > 0 and "n_weighted_features" in ok_fold_df.columns
            else np.nan
        ),
        "selected_feature_weight_modes": (
            ",".join(sorted(ok_fold_df["feature_weight_mode"].dropna().astype(str).unique()))
            if len(ok_fold_df) > 0 and "feature_weight_mode" in ok_fold_df.columns
            else ""
        ),
        "selected_feature_selection_modes": (
            ",".join(sorted(ok_fold_df["feature_selection_mode"].dropna().astype(str).unique()))
            if len(ok_fold_df) > 0 and "feature_selection_mode" in ok_fold_df.columns
            else ""
        ),

        "mean_valid_y_sst": (
            float(ok_fold_df["valid_y_sst"].mean())
            if len(ok_fold_df) > 0 and "valid_y_sst" in ok_fold_df.columns
            else np.nan
        ),
        "min_valid_y_sst": (
            float(ok_fold_df["valid_y_sst"].min())
            if len(ok_fold_df) > 0 and "valid_y_sst" in ok_fold_df.columns
            else np.nan
        ),
        "mean_valid_y_range": (
            float(ok_fold_df["valid_y_range"].mean())
            if len(ok_fold_df) > 0 and "valid_y_range" in ok_fold_df.columns
            else np.nan
        ),
        "n_low_sst_folds": (
            int((ok_fold_df["valid_y_sst"] < LOW_VALID_SST_WARN_THRESHOLD).sum())
            if len(ok_fold_df) > 0 and "valid_y_sst" in ok_fold_df.columns
            else 0
        ),
        "mean_valid_r2_excluding_low_sst": (
            float(ok_fold_df.loc[
                ok_fold_df["valid_y_sst"] >= LOW_VALID_SST_WARN_THRESHOLD,
                "valid_r2"
            ].mean())
            if (
                len(ok_fold_df) > 0
                and "valid_y_sst" in ok_fold_df.columns
                and "valid_r2" in ok_fold_df.columns
                and (ok_fold_df["valid_y_sst"] >= LOW_VALID_SST_WARN_THRESHOLD).any()
            )
            else np.nan
        ),
    }

    return {
        "fold_df": pd.DataFrame(fold_rows),
        "feature_df": pd.concat(feature_rows, ignore_index=True) if feature_rows else pd.DataFrame(),
        "feature_score_df": pd.concat(feature_score_rows, ignore_index=True) if feature_score_rows else pd.DataFrame(),
        "feature_weight_df": pd.concat(feature_weight_rows, ignore_index=True) if feature_weight_rows else pd.DataFrame(),
        "search_df": pd.concat(search_rows_all, ignore_index=True) if search_rows_all else pd.DataFrame(),
        "pred_df": pred_df,
        "metrics": metrics,
    }


In [46]:
from itertools import product
import numpy as np

def generate_weight_grid(n_models: int, step: float = 0.1):
    """
    Generate nonnegative weight combinations that sum to 1.
    Example for n_models=2 and step=0.1:
    [0.0, 1.0], [0.1, 0.9], ..., [1.0, 0.0]
    """
    if n_models < 2:
        raise ValueError("n_models must be at least 2")

    step_count = int(round(1.0 / step))
    if not np.isclose(step * step_count, 1.0):
        raise ValueError("step must divide 1 exactly, e.g. 0.5, 0.25, 0.2, 0.1")

    grid = []
    for ints in product(range(step_count + 1), repeat=n_models):
        if sum(ints) == step_count:
            weights = [x * step for x in ints]
            grid.append(weights)

    return grid

In [47]:
# ==========================================
# Late-fusion search on train OOF         
# ==========================================
def search_best_late_fusion_weights(
    oof_train_dict: dict[str, np.ndarray],
    y_train: pd.Series,
):
    """
    oof_train_dict example:
      {
        "A": oof_pred_from_atomic_A,
        "B": oof_pred_from_atomic_B,
        "C": oof_pred_from_atomic_C,
      }

    We search fusion weights using ONLY train-fold OOF predictions.
    This avoids looking at outer validation labels when choosing weights.
    """
    keys = list(oof_train_dict.keys())
    y_true = y_train.values

    rows = []

    if len(keys) == 2:
        for w in WEIGHT_GRID_2:
            pred = w[0] * oof_train_dict[keys[0]] + w[1] * oof_train_dict[keys[1]]
            rows.append({
                "weights": w,
                "train_oof_r2": r2_score(y_true, pred),
                "train_oof_mae": mean_absolute_error(y_true, pred),
            })

    elif len(keys) == 3:
        for w in WEIGHT_GRID_3:
            pred = (
                w[0] * oof_train_dict[keys[0]]
                + w[1] * oof_train_dict[keys[1]]
                + w[2] * oof_train_dict[keys[2]]
            )
            rows.append({
                "weights": w,
                "train_oof_r2": r2_score(y_true, pred),
                "train_oof_mae": mean_absolute_error(y_true, pred),
            })

    else:
        raise ValueError("Late fusion search currently supports 2 or 3 atomic inputs.")

    weight_df = pd.DataFrame(rows).sort_values(
        by=["train_oof_r2", "train_oof_mae"],
        ascending=[False, True]
    ).reset_index(drop=True)

    best_row = weight_df.iloc[0].to_dict()
    return best_row, weight_df

In [48]:
def run_early_fusion_cv_on_predefined_train(
    tissue: str,
    combo: str,
    tp: int,
    y_all: pd.Series,
    train_ids: list[str],
    model_type: str,
    clinical_mode: str = "light",
):
    """
    Run EF CV only on predefined train patients.

    Important:
    - Do not create a separate X_train here for modeling.
    - Use train_ids only to define allowed_ids.
    - Actual loading/filtering/CV happens inside run_early_fusion_cv().
    """
    train_ids = [str(pid) for pid in train_ids]
    y_all_index = y_all.index.astype(str)
    y_all = y_all.copy()
    y_all.index = y_all_index

    train_ids_in = [pid for pid in train_ids if pid in y_all.index]

    if len(train_ids_in) < N_SPLITS_OUTER:
        raise ValueError(
            f"Not enough train IDs available in target: "
            f"n_train_ids_in={len(train_ids_in)}, n_splits={N_SPLITS_OUTER}"
        )

    cv_result = run_early_fusion_cv(
        tissue=tissue,
        combo=combo,
        tp=tp,
        y_all=y_all,
        model_type=model_type,
        clinical_mode=clinical_mode,
        allowed_ids=train_ids_in,
    )

    fold_df = cv_result["fold_df"]

    return {
        "mode": "cv_train_only",
        "model_type": model_type,
        "clinical_mode": clinical_mode,
        "n_train": len(train_ids_in),
        "cv_fold_df": fold_df,
        "cv_feature_df": cv_result["feature_df"],
        "cv_feature_score_df": cv_result["feature_score_df"],
        "cv_feature_weight_df": cv_result["feature_weight_df"],
        "cv_search_df": cv_result["search_df"],
        "cv_pred_df": cv_result["pred_df"],
        "metrics": cv_result["metrics"],
        "oof_r2": cv_result["metrics"]["oof_r2"],
        "oof_mae": cv_result["metrics"]["oof_mae"],
        "cv_r2": cv_result["metrics"]["oof_r2"],
        "cv_mae": cv_result["metrics"]["oof_mae"],
    }


In [49]:
def run_one_ef_experiment(
    tissue: str,
    combo: str,
    tp: int,
    y_all: pd.Series,
    train_ids: list[str],
    model_type: str,
    clinical_mode: str = "light",
):
    try:
        ef_result = run_early_fusion_cv_on_predefined_train(
            tissue=tissue,
            combo=combo,
            tp=tp,
            y_all=y_all,
            train_ids=train_ids,
            model_type=model_type,
            clinical_mode=clinical_mode,
        )

        ef_fold_df = ef_result["cv_fold_df"]
        metrics = ef_result["metrics"]

        # Save detailed EF outputs for debugging/comparison
        detail_dir = SAVE_DIR / "ef_detail_outputs"
        detail_dir.mkdir(parents=True, exist_ok=True)
        prefix = f"ef_{tissue}_{combo}_tp{tp}_{model_type}_{clinical_mode}"

        if isinstance(ef_fold_df, pd.DataFrame) and len(ef_fold_df) > 0:
            ef_fold_df.to_csv(detail_dir / f"{prefix}_fold_metrics.csv", index=False)

        if isinstance(ef_result.get("cv_feature_df", None), pd.DataFrame) and len(ef_result["cv_feature_df"]) > 0:
            ef_result["cv_feature_df"].to_csv(detail_dir / f"{prefix}_selected_features.csv", index=False)

        if isinstance(ef_result.get("cv_feature_score_df", None), pd.DataFrame) and len(ef_result["cv_feature_score_df"]) > 0:
            ef_result["cv_feature_score_df"].to_csv(detail_dir / f"{prefix}_feature_scores.csv", index=False)

        if isinstance(ef_result.get("cv_feature_weight_df", None), pd.DataFrame) and len(ef_result["cv_feature_weight_df"]) > 0:
            ef_result["cv_feature_weight_df"].to_csv(detail_dir / f"{prefix}_feature_weights.csv", index=False)

        if isinstance(ef_result.get("cv_search_df", None), pd.DataFrame) and len(ef_result["cv_search_df"]) > 0:
            ef_result["cv_search_df"].to_csv(detail_dir / f"{prefix}_search_grid.csv", index=False)

        if isinstance(ef_result.get("cv_pred_df", None), pd.DataFrame) and len(ef_result["cv_pred_df"]) > 0:
            ef_result["cv_pred_df"].to_csv(detail_dir / f"{prefix}_oof_predictions.csv", index=False)

        return pd.DataFrame([{
            "status": metrics.get("status", "ok"),
            "fusion_type": "EF",
            "experiment_tag": EXPERIMENT_TAG,
            "outer_cv_type": metrics.get("outer_cv_type", ""),
            "use_feature_weighting": metrics.get("use_feature_weighting", USE_FEATURE_WEIGHTING),
            "feature_weight_modes": metrics.get("feature_weight_modes", ",".join(FEATURE_WEIGHT_MODES)),
            "feature_weight_min": metrics.get("feature_weight_min", FEATURE_WEIGHT_MIN),
            "feature_weight_max": metrics.get("feature_weight_max", FEATURE_WEIGHT_MAX),
            "selected_feature_weight_modes": metrics.get("selected_feature_weight_modes", ""),
            "selected_feature_selection_modes": metrics.get("selected_feature_selection_modes", ""),
            "model_type": model_type,
            "clinical_mode": clinical_mode,
            "tuning_group": get_ef_tuning_group(tissue, tp),
            "tissue": tissue,
            "combo": combo,
            "tp": tp,
            "use_delta_view": USE_DELTA_VIEW,
            "n_train": ef_result["n_train"],

            "oof_r2": metrics.get("oof_r2", np.nan),
            "oof_mae": metrics.get("oof_mae", np.nan),
            "cv_r2": metrics.get("cv_r2", np.nan),
            "cv_mae": metrics.get("cv_mae", np.nan),

            "mean_valid_r2": metrics.get("mean_valid_r2", np.nan),
            "median_valid_r2": metrics.get("median_valid_r2", np.nan),
            "min_valid_r2": metrics.get("min_valid_r2", np.nan),
            "valid_r2_q25": metrics.get("valid_r2_q25", np.nan),
            "valid_r2_q75": metrics.get("valid_r2_q75", np.nan),
            "mean_valid_mae": metrics.get("mean_valid_mae", np.nan),

            "mean_train_r2": metrics.get("mean_train_r2", np.nan),
            "mean_train_mae": metrics.get("mean_train_mae", np.nan),
            "mean_inner_cv_r2": metrics.get("mean_inner_cv_r2", np.nan),
            "mean_feature_weight": metrics.get("mean_feature_weight", np.nan),
            "mean_max_feature_weight": metrics.get("mean_max_feature_weight", np.nan),
            "mean_n_weighted_features": metrics.get("mean_n_weighted_features", np.nan),

            "mean_valid_y_sst": metrics.get("mean_valid_y_sst", np.nan),
            "min_valid_y_sst": metrics.get("min_valid_y_sst", np.nan),
            "mean_valid_y_range": metrics.get("mean_valid_y_range", np.nan),
            "n_low_sst_folds": metrics.get("n_low_sst_folds", 0),
            "mean_valid_r2_excluding_low_sst": metrics.get("mean_valid_r2_excluding_low_sst", np.nan),

            "n_total_samples": metrics.get("n_total_samples", np.nan),
            "n_pred_samples": metrics.get("n_pred_samples", np.nan),
            "pred_coverage": metrics.get("pred_coverage", np.nan),
            "n_total_folds": metrics.get("n_total_folds", np.nan),
            "n_completed_folds": metrics.get("n_completed_folds", np.nan),
            "n_failed_folds": metrics.get("n_failed_folds", np.nan),

            "error_message": None,
        }])

    except Exception as e:
        return pd.DataFrame([{
            "status": "failed",
            "fusion_type": "EF",
            "experiment_tag": EXPERIMENT_TAG,
            "outer_cv_type": "",
            "use_feature_weighting": USE_FEATURE_WEIGHTING,
            "feature_weight_modes": ",".join(FEATURE_WEIGHT_MODES),
            "feature_weight_min": FEATURE_WEIGHT_MIN,
            "feature_weight_max": FEATURE_WEIGHT_MAX,
            "selected_feature_weight_modes": "",
            "selected_feature_selection_modes": "",
            "model_type": model_type,
            "clinical_mode": clinical_mode,
            "tuning_group": get_ef_tuning_group(tissue, tp),
            "tissue": tissue,
            "combo": combo,
            "tp": tp,
            "use_delta_view": USE_DELTA_VIEW,
            "n_train": np.nan,

            "oof_r2": np.nan,
            "oof_mae": np.nan,
            "cv_r2": np.nan,
            "cv_mae": np.nan,

            "mean_valid_r2": np.nan,
            "median_valid_r2": np.nan,
            "min_valid_r2": np.nan,
            "valid_r2_q25": np.nan,
            "valid_r2_q75": np.nan,
            "mean_valid_mae": np.nan,

            "mean_train_r2": np.nan,
            "mean_train_mae": np.nan,
            "mean_inner_cv_r2": np.nan,
            "mean_feature_weight": np.nan,
            "mean_max_feature_weight": np.nan,
            "mean_n_weighted_features": np.nan,

            "mean_valid_y_sst": np.nan,
            "min_valid_y_sst": np.nan,
            "mean_valid_y_range": np.nan,
            "n_low_sst_folds": 0,
            "mean_valid_r2_excluding_low_sst": np.nan,

            "n_total_samples": np.nan,
            "n_pred_samples": np.nan,
            "pred_coverage": np.nan,
            "n_total_folds": np.nan,
            "n_completed_folds": 0,
            "n_failed_folds": np.nan,

            "error_message": str(e),
        }])


In [50]:
y_all = load_target(TARGET_PATH)

jobs = []
for tissue, combo, tp in product(TISSUES, COMBOS, TIMEPOINTS):
    for clinical_mode in get_ef_clinical_modes_for_job(tissue, tp):
        for model_type in get_ef_model_types_for_job(tissue, tp):
            jobs.append({
                "tissue": tissue,
                "combo": combo,
                "tp": tp,
                "model_type": model_type,
                "clinical_mode": clinical_mode,
                "tuning_group": get_ef_tuning_group(tissue, tp),
            })

print("n_jobs:", len(jobs))

job_manifest_df = pd.DataFrame(jobs)
job_manifest_path = SAVE_DIR / f"job_manifest_{EXPERIMENT_TAG}.csv"
job_manifest_df.to_csv(job_manifest_path, index=False)
print("job_manifest_path:", job_manifest_path)

all_result_dfs = Parallel(
    n_jobs=N_JOBS_EXPERIMENT,
    backend=PARALLEL_BACKEND,
    verbose=10,
)(
    delayed(run_one_ef_experiment)(
        tissue=job["tissue"],
        combo=job["combo"],
        tp=job["tp"],
        y_all=y_all,
        train_ids=TRAIN_IDS,
        model_type=job["model_type"],
        clinical_mode=job["clinical_mode"],
    )
    for job in jobs
)

summary_df = pd.concat(all_result_dfs, ignore_index=True)

summary_df = summary_df.sort_values(
    by=["status", "oof_r2", "mean_valid_r2", "median_valid_r2", "min_valid_r2", "oof_mae"],
    ascending=[True, False, False, False, False, True],
    na_position="last",
).reset_index(drop=True)


n_jobs: 44
job_manifest_path: ../../DifferentCom_data_rebuild/results_multi_omics_ef_train_only_ef_paper_style_v13_tp_model_refined_tuning/job_manifest_ef_paper_style_v13_tp_model_refined_tuning.csv


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


[ROW-FILTER][EF] tissue=csf combo=ABC tp=24 clinical=omics_only fold=1 | train_before=44 | train_after=35 | removed_no_omics=9
[ROW-FILTER][EF] tissue=csf combo=ABC tp=48 clinical=omics_only fold=1 | train_before=44 | train_after=34 | removed_no_omics=10
[ROW-FILTER][EF] tissue=csf combo=ABC tp=24 clinical=light fold=1 | train_before=44 | train_after=35 | removed_no_omics=9
[ROW-FILTER][EF] tissue=csf combo=ABC tp=48 clinical=omics_only fold=1 | train_before=44 | train_after=34 | removed_no_omics=10
[ROW-FILTER][EF] tissue=csf combo=ABC tp=24 clinical=light fold=1 | train_before=44 | train_after=35 | removed_no_omics=9
[ROW-FILTER][EF] tissue=csf combo=ABC tp=24 clinical=omics_only fold=1 | train_before=44 | train_after=35 | removed_no_omics=9
[ROW-FILTER][EF] tissue=csf combo=ABC tp=48 clinical=light fold=1 | train_before=44 | train_after=34 | removed_no_omics=10
[ROW-FILTER][EF] tissue=csf combo=ABC tp=48 clinical=light fold=1 | train_before=44 | train_after=34 | removed_no_omics=10


[Parallel(n_jobs=8)]: Done   2 tasks      | elapsed:   28.4s


[DEBUG][BLOCKWISE] counts = {'PROT': 256, 'MET': 459, 'RNA': 258, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 250, 'MET': 459, 'RNA': 269, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 249, 'MET': 459, 'RNA': 242, 'CLIN': 13}
[DEBUG][BLOCKWISE] counts = {'PROT': 250, 'MET': 459, 'RNA': 269, 'CLIN': 13}
[DEBUG][BLOCKWISE] counts = {'PROT': 243, 'MET': 461, 'RNA': 154, 'CLIN': 14}
[DEBUG][BLOCKWISE] counts = {'PROT': 233, 'MET': 461, 'RNA': 146, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 242, 'MET': 461, 'RNA': 174, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 256, 'MET': 459, 'RNA': 258, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 250, 'MET': 459, 'RNA': 269, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 233, 'MET': 461, 'RNA': 146, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 249, 'MET': 459, 'RNA': 242, 'CLIN': 12}
[DEBUG][BLOCKWISE] counts = {'PROT': 250, 'MET': 459, 'RNA': 269, 'CLIN': 15}
[DEBUG][BLOCKWISE] counts = {'PROT': 243, 'MET': 461, 'RNA': 154, 'CLIN

...

[Parallel(n_jobs=8)]: Done  34 out of  44 | elapsed:  5.4min remaining:  1.6min


[DEBUG][BLOCKWISE] counts = {'PROT': 132, 'MET': 275, 'RNA': 219, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 132, 'MET': 275, 'RNA': 219, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 132, 'MET': 275, 'RNA': 213, 'CLIN': 15}
[DEBUG][BLOCKWISE] counts = {'PROT': 131, 'MET': 274, 'RNA': 295, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 131, 'MET': 273, 'RNA': 285, 'CLIN': 14}
[DEBUG][BLOCKWISE] counts = {'PROT': 132, 'MET': 275, 'RNA': 219, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 131, 'MET': 273, 'RNA': 285, 'CLIN': 17}
[DEBUG][BLOCKWISE] counts = {'PROT': 218, 'MET': 461, 'RNA': 119, 'CLIN': 15}
[DEBUG][BLOCKWISE] counts = {'PROT': 132, 'MET': 275, 'RNA': 213, 'CLIN': 16}
[DEBUG][BLOCKWISE] counts = {'PROT': 132, 'MET': 275, 'RNA': 219, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 131, 'MET': 274, 'RNA': 295, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 132, 'MET': 275, 'RNA': 219, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 131, 'MET': 273, 'RNA': 285, 'CLIN

[Parallel(n_jobs=8)]: Done  39 out of  44 | elapsed:  6.8min remaining:   52.4s


[DEBUG][BLOCKWISE] counts = {'PROT': 131, 'MET': 274, 'RNA': 280, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 132, 'MET': 275, 'RNA': 213, 'CLIN': 16}
[DEBUG][BLOCKWISE] counts = {'PROT': 132, 'MET': 274, 'RNA': 286, 'CLIN': 14}
[DEBUG][BLOCKWISE] counts = {'PROT': 131, 'MET': 274, 'RNA': 280, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 132, 'MET': 275, 'RNA': 213, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 132, 'MET': 275, 'RNA': 213, 'CLIN': 15}
[DEBUG][BLOCKWISE] counts = {'PROT': 131, 'MET': 275, 'RNA': 217, 'CLIN': 16}
[DEBUG][BLOCKWISE] counts = {'PROT': 132, 'MET': 274, 'RNA': 286, 'CLIN': 14}
[DEBUG][BLOCKWISE] counts = {'PROT': 131, 'MET': 274, 'RNA': 280, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 132, 'MET': 275, 'RNA': 213, 'CLIN': 15}
[DEBUG][BLOCKWISE] counts = {'PROT': 132, 'MET': 275, 'RNA': 213, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 131, 'MET': 274, 'RNA': 280, 'CLIN': 0}
[DEBUG][BLOCKWISE] counts = {'PROT': 132, 'MET': 274, 'RNA': 286, 'CLI

[Parallel(n_jobs=8)]: Done  44 out of  44 | elapsed:  8.7min finished


In [51]:
# ===============================
# FINAL SUMMARY PRINT + SAVE
# ===============================

print("\n===== FINAL SUMMARY (ALL SORTED) =====")
print(
    summary_df.sort_values(
        by=["model_type", "tissue", "tp"],
        ascending=[True, True, True]
    ).reset_index(drop=True)
)

summary_path = SAVE_DIR / f"multi_omics_ef_summary_{EXPERIMENT_TAG}.csv"
summary_df.to_csv(summary_path, index=False)


===== FINAL SUMMARY (ALL SORTED) =====
   status fusion_type                              experiment_tag  \
0      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
1      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
2      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
3      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
4      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
5      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
6      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
7      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
8      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
9      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
10     ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
11     ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
12     ok          EF  ef_paper_style_v13_tp_model_refined_tuni

In [52]:
# ===============================
# GROUPED SUMMARY BY TISSUE / TP / MODEL / CLINICAL MODE
# ===============================

ok_df = summary_df[summary_df["status"].eq("ok")].copy()

if len(ok_df) > 0:
    grouped_dir = SAVE_DIR / "grouped_summary"
    grouped_dir.mkdir(parents=True, exist_ok=True)

    def summarize_group(group_cols):
        return (
            ok_df
            .groupby(group_cols, dropna=False)
            .agg(
                n=("oof_r2", "size"),
                mean_oof_r2=("oof_r2", "mean"),
                median_oof_r2=("oof_r2", "median"),
                best_oof_r2=("oof_r2", "max"),
                mean_valid_r2=("mean_valid_r2", "mean"),
                median_valid_r2=("median_valid_r2", "median"),
                mean_train_r2=("mean_train_r2", "mean"),
                mean_oof_mae=("oof_mae", "mean"),
            )
            .reset_index()
            .sort_values(["mean_oof_r2", "best_oof_r2"], ascending=[False, False])
        )

    tissue_summary = summarize_group(["tissue"])
    tissue_tp_summary = summarize_group(["tissue", "tp"])
    tissue_model_summary = summarize_group(["tissue", "model_type"])
    tissue_tp_model_summary = summarize_group(["tissue", "tp", "model_type"])
    tissue_tp_clinical_summary = summarize_group(["tissue", "tp", "clinical_mode"])
    tissue_tp_model_clinical_summary = summarize_group(["tissue", "tp", "model_type", "clinical_mode"])
    tuning_group_summary = summarize_group(["tuning_group"])
    tuning_group_model_summary = summarize_group(["tuning_group", "model_type"])

    tissue_summary.to_csv(grouped_dir / f"grouped_by_tissue_{EXPERIMENT_TAG}.csv", index=False)
    tissue_tp_summary.to_csv(grouped_dir / f"grouped_by_tissue_tp_{EXPERIMENT_TAG}.csv", index=False)
    tissue_model_summary.to_csv(grouped_dir / f"grouped_by_tissue_model_{EXPERIMENT_TAG}.csv", index=False)
    tissue_tp_model_summary.to_csv(grouped_dir / f"grouped_by_tissue_tp_model_{EXPERIMENT_TAG}.csv", index=False)
    tissue_tp_clinical_summary.to_csv(grouped_dir / f"grouped_by_tissue_tp_clinical_{EXPERIMENT_TAG}.csv", index=False)
    tissue_tp_model_clinical_summary.to_csv(grouped_dir / f"grouped_by_tissue_tp_model_clinical_{EXPERIMENT_TAG}.csv", index=False)
    tuning_group_summary.to_csv(grouped_dir / f"grouped_by_tuning_group_{EXPERIMENT_TAG}.csv", index=False)
    tuning_group_model_summary.to_csv(grouped_dir / f"grouped_by_tuning_group_model_{EXPERIMENT_TAG}.csv", index=False)

    print("\n===== GROUPED SUMMARY: TUNING GROUP =====")
    print(tuning_group_summary)

    print("\n===== GROUPED SUMMARY: TUNING GROUP + MODEL =====")
    print(tuning_group_model_summary)

    print("\n===== GROUPED SUMMARY: TISSUE =====")
    print(tissue_summary)

    print("\n===== GROUPED SUMMARY: TISSUE + TP =====")
    print(tissue_tp_summary)

    print("\n===== GROUPED SUMMARY: TISSUE + TP + MODEL =====")
    print(tissue_tp_model_summary)

    print("\n===== GROUPED SUMMARY: TISSUE + TP + CLINICAL MODE =====")
    print(tissue_tp_clinical_summary)

    print("\n===== GROUPED SUMMARY: TISSUE + TP + MODEL + CLINICAL MODE =====")
    print(tissue_tp_model_clinical_summary)


    # Best row per tissue/TP to check whether the result follows paper-style tissue/timepoint interpretation.
    best_by_tissue_tp = (
        ok_df
        .sort_values(["tissue", "tp", "oof_r2", "mean_valid_r2"], ascending=[True, True, False, False])
        .groupby(["tissue", "tp"], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    keep_cols = [
        "tissue", "tp", "model_type", "clinical_mode",
        "oof_r2", "mean_valid_r2", "median_valid_r2", "min_valid_r2",
        "mean_train_r2", "selected_feature_selection_modes",
        "selected_feature_weight_modes",
    ]
    keep_cols = [c for c in keep_cols if c in best_by_tissue_tp.columns]
    best_by_tissue_tp[keep_cols].to_csv(
        grouped_dir / f"best_row_by_tissue_tp_{EXPERIMENT_TAG}.csv",
        index=False,
    )

    print("\n===== BEST ROW BY TISSUE + TP =====")
    print(best_by_tissue_tp[keep_cols])


    # Paper-style primary result table: one best model per tissue + TP.
    # Use this table as the main interpretation target instead of one global mean OOF.
    paper_style_best_table = best_by_tissue_tp[keep_cols].copy()
    paper_style_best_table = paper_style_best_table.sort_values(["tissue", "tp"]).reset_index(drop=True)
    paper_style_best_table.to_csv(
        grouped_dir / f"paper_style_best_by_tissue_tp_{EXPERIMENT_TAG}.csv",
        index=False,
    )

    print("\n===== PAPER-STYLE PRIMARY TABLE: BEST MODEL BY TISSUE + TP =====")
    print(paper_style_best_table)

else:
    print("No ok rows available for grouped summary.")



===== GROUPED SUMMARY: TUNING GROUP =====
  tuning_group   n  mean_oof_r2  median_oof_r2  best_oof_r2  mean_valid_r2  \
1       strong  24     0.150635       0.157676     0.217536       0.139814   
0       middle  14     0.049162       0.048183     0.089067       0.027794   
2         weak   6    -0.027795      -0.002549     0.003662      -0.019201   

   median_valid_r2  mean_train_r2  mean_oof_mae  
1         0.117065       0.640604     14.832023  
0         0.112673       0.702114     15.710286  
2         0.004533       0.556087     17.178063  

===== GROUPED SUMMARY: TUNING GROUP + MODEL =====
  tuning_group  model_type  n  mean_oof_r2  median_oof_r2  best_oof_r2  \
4       strong         gbr  4     0.187745       0.187652     0.217536   
7       strong  svr_linear  4     0.184614       0.188273     0.204700   
3       strong  elasticnet  6     0.149461       0.147128     0.214499   
5       strong         pls  4     0.137573       0.146702     0.188541   
6       strong       ri

In [53]:
for mt in MODEL_TYPES:
    print(f"\n===== SUMMARY: {mt} =====")
    print(
        summary_df[summary_df["model_type"] == mt]
        .sort_values(by=["tissue", "tp"])
        .reset_index(drop=True)
    )



===== SUMMARY: ridge =====
   status fusion_type                              experiment_tag  \
0      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
1      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
2      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
3      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
4      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
5      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
6      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
7      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
8      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
9      ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
10     ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
11     ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
12     ok          EF  ef_paper_style_v13_tp_model_refined_tuning   
13    